# LangGraph Agentic RAG — SEC EDGAR v2

**Changes from v1:**
- **Dataset**: Uses local `sec_chunks.jsonl` (12 companies, 155 filings, 2023–2026) instead of FinanceBench
- **Proposal 1 — Query Decomposition**: A `Planner` node decomposes multi-period / cross-company questions into 2–3 sub-queries, each with optional `ticker`/`filing_year` metadata filters for targeted retrieval
- **Proposal 2 — Refined Critic**: `CriticOutput.decision` now has a third state `insufficient_data` — the agent safely exits when required data is absent from the filing rather than looping

In [1]:
import os
import re
import time
import json
import random
import warnings
import threading
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, TypedDict, Literal, Tuple, Set
from collections import deque, Counter

import chromadb
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

from pydantic import BaseModel, Field, field_validator, model_validator

from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

# Resolve project root robustly when notebook runs from subfolder
import sys

def detect_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "config.py").exists() and (p / "shared_retriever.py").exists():
            return p
    return start

PROJECT_ROOT = detect_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config as config_module
from config import CONFIG as _CONFIG, resolve_model_name, get_provider_order, resolve_fallback_model_names
from shared_retriever import initialize_corpus, CorpusIndex

warnings.filterwarnings("ignore")

c:\Users\jeeey\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loading embedding model: nomic-ai/nomic-embed-text-v1.5...


<All keys matched successfully>
c:\Users\jeeey\GenAI-with-LLMs\shared_retriever.py:118: RuntimeWarning: trust_remote_code=True for dense model 'nomic-ai/nomic-embed-text-v1.5'. Only enable this for trusted model repositories.
  warnings.warn(


Loading reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
# Non-interactive API key loading for notebook automation
# Gemini-only configuration: read GEMINI_API_KEY from the environment and
# pass that single value into ChatGoogleGenerativeAI internally.
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "").strip()
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()

if not GEMINI_API_KEY:
    print("[WARN] GEMINI_API_KEY is not set. Gemini LLM calls may fail later.")
else:
    print("Gemini API key loaded from GEMINI_API_KEY.")


Gemini API key loaded from GEMINI_API_KEY.


In [3]:
# Fixed Gemini model selection for this notebook run.
# Use Gemini 2.5 Flash (non-lite) for all roles without prompting.
SELECTED_GEMINI_PRESET = {
    "label": "Flash for all roles",
    "generator": "gemini-2.5-flash",
    "agent": "gemini-2.5-flash",
    "judge": "gemini-2.5-flash",
}
print(
    f"Selected Gemini preset: {SELECTED_GEMINI_PRESET['label']} "
    f"(gen={SELECTED_GEMINI_PRESET['generator']}, "
    f"agent={SELECTED_GEMINI_PRESET['agent']}, "
    f"judge={SELECTED_GEMINI_PRESET['judge']})"
)


Selected Gemini preset: Flash for all roles (gen=gemini-2.5-flash, agent=gemini-2.5-flash, judge=gemini-2.5-flash)


In [4]:
# Import centralized CONFIG from config.py
CONFIG = _CONFIG.copy()

# Align LangGraph LLM selection with advanced_rag_upgrade_v2 805pm.ipynb
ADVANCED_RAG_MODEL_ALIGNMENT = {
    "provider": "gemini",
    "provider_fallback_order": ["gemini"],
    "execution_profile": "dev",
    "gemini_dev_generator_model": SELECTED_GEMINI_PRESET["generator"],
    "gemini_dev_agent_model": SELECTED_GEMINI_PRESET["agent"],
    "gemini_dev_judge_model": SELECTED_GEMINI_PRESET["judge"],
    "gemini_final_generator_model": SELECTED_GEMINI_PRESET["generator"],
    "gemini_final_agent_model": SELECTED_GEMINI_PRESET["agent"],
    "gemini_final_judge_model": SELECTED_GEMINI_PRESET["judge"],
    "gemini_fallback_generator_models": [],
    "gemini_fallback_agent_models": [],
    "gemini_fallback_judge_models": [],
    "provider_rpm": {"gemini": 10},
}

# Add LangGraph-specific config extensions
LANGGRAPH_SPECIFIC = {
    "random_seed": 42,
    "use_pilot": False,
    "pilot_n_questions": 5,
    "full_n_questions": 100,
    "use_llm_judge": True,
    "use_few_shot_examples": True,
    "pilot_judge_sample_n": 1,
    "full_judge_sample_n": 2,
    "train_split": "dev",
    "eval_split": "test",
    "auto_export_results_input": False,
    "verbose_eval_logging": True,
    "verbose_drift_logging": False,
    "verbose_drift_failures": True,
    "max_context_retries": 1,
    "max_repair_retrievals": 1,
    "multi_query_n": 2,
    "rrf_k": 60,
    "compress_top_sentences": 4,
    "decomposition_top_k_per_subquery": 4,
    "enable_retrieval_sanity_check": True,
    "inter_question_sleep_sec": 0.5,
    "reuse_prior_advanced_results": True,
    "require_prior_advanced_results": False,
    "prior_advanced_results_path": str((PROJECT_ROOT / "baseline_advanced_rag" / "results" / "advanced_results.csv").resolve()),
}

CONFIG.update(ADVANCED_RAG_MODEL_ALIGNMENT)
CONFIG.update(LANGGRAPH_SPECIFIC)

# Keep config.py helper functions in sync with the notebook override.
# resolve_model_name()/resolve_fallback_model_names() read config_module.CONFIG,
# so we update that module-level state as well.
config_module.CONFIG.update(CONFIG)
CONFIG["max_rpm"] = CONFIG["provider_rpm"].get(CONFIG["provider"], CONFIG.get("max_rpm", 10))
CONFIG["judge_sample_n"] = (
    CONFIG["pilot_judge_sample_n"] if CONFIG["use_pilot"] else CONFIG["full_judge_sample_n"]
) if CONFIG["use_llm_judge"] else 0
config_module.CONFIG["max_rpm"] = CONFIG["max_rpm"]
config_module.CONFIG["judge_sample_n"] = CONFIG["judge_sample_n"]

print(f'Config loaded. Provider: {CONFIG["provider"]}, Profile: {CONFIG["execution_profile"]}')
print(
    'Advanced-RAG model alignment: '
    f'generator={resolve_model_name("generator", provider="gemini")}, '
    f'agent={resolve_model_name("agent", provider="gemini")}, '
    f'judge={resolve_model_name("judge", provider="gemini")}'
)
print(f'Prior advanced results reuse: {CONFIG["reuse_prior_advanced_results"]} | path={CONFIG["prior_advanced_results_path"]}')

Config loaded. Provider: gemini, Profile: dev
Advanced-RAG model alignment: generator=gemini-2.5-flash, agent=gemini-2.5-flash, judge=gemini-2.5-flash
Prior advanced results reuse: True | path=C:\Users\jeeey\GenAI-with-LLMs\baseline_advanced_rag\results\advanced_results.csv


In [5]:
# ---------------------------------------------------------------------------
# Evaluation dataset from sec_rag_team_share/evaluation/GenAI Eval QA.csv
# Use the 'dev' split for prompt development / training-style iteration and
# the 'test' split for evaluation.
# ---------------------------------------------------------------------------

raw_eval_df = pd.read_csv(CONFIG["sec_eval_csv_path"])
raw_eval_df = raw_eval_df[raw_eval_df["question_id"].notna()].copy()
raw_eval_df["question_id"] = raw_eval_df["question_id"].astype(str).str.strip()
raw_eval_df = raw_eval_df[raw_eval_df["question_id"] != ""].copy()

full_eval_df = raw_eval_df.rename(columns={
    "question_id": "id",
    "expected_answer": "reference_answer",
}).copy()

full_eval_df["question_number"] = pd.to_numeric(full_eval_df["id"], errors="coerce")
full_eval_df["id"] = full_eval_df["id"].apply(lambda x: f"SEC_{int(float(x)):03d}" if str(x).replace(".", "", 1).isdigit() else str(x))
full_eval_df["expected_decision"] = full_eval_df["expected_decision"].fillna("ANSWER").astype(str).str.strip().str.lower()
full_eval_df["question_type"] = full_eval_df["question_type"].fillna("extractive").astype(str).str.strip()
full_eval_df["company"] = full_eval_df["company"].fillna("").astype(str).str.strip()
full_eval_df["reference_answer"] = full_eval_df["reference_answer"].fillna("").astype(str).str.strip()
full_eval_df["split"] = full_eval_df["split"].fillna("").astype(str).str.strip().str.lower()

train_df = full_eval_df[full_eval_df["split"] == CONFIG["train_split"]].reset_index(drop=True)
eval_source_df = full_eval_df[full_eval_df["split"] == CONFIG["eval_split"]].reset_index(drop=True)

if CONFIG["use_pilot"]:
    eval_df = eval_source_df.sample(
        n=min(CONFIG["pilot_n_questions"], len(eval_source_df)),
        random_state=CONFIG["random_seed"],
    ).sort_values("id").reset_index(drop=True)
    print(f"PILOT RUN: {len(eval_df)} questions from split='{CONFIG['eval_split']}'")
else:
    eval_df = eval_source_df.sample(
        n=min(CONFIG["full_n_questions"], len(eval_source_df)),
        random_state=CONFIG["random_seed"],
    ).sort_values("id").reset_index(drop=True)
    print(f"FULL RUN: {len(eval_df)} questions from split='{CONFIG['eval_split']}'")

print(f"Training/dev split size: {len(train_df)} | Evaluation/test split size: {len(eval_source_df)}")
print(eval_df[["id", "question_type", "company", "ticker", "form_type", "filing_year"]].to_string(index=False))

FULL RUN: 80 questions from split='test'
Training/dev split size: 20 | Evaluation/test split size: 80
     id question_type            company ticker form_type  filing_year
SEC_006    extractive             NVIDIA   NVDA      10-K       2024.0
SEC_007    extractive             NVIDIA   NVDA      10-K       2025.0
SEC_008    extractive     JPMorgan Chase    JPM      10-K       2023.0
SEC_009    extractive     JPMorgan Chase    JPM      10-K       2024.0
SEC_010    extractive    Bank of America    BAC      10-Q       2025.0
SEC_011    extractive    Bank of America    BAC      10-K       2024.0
SEC_012    extractive Berkshire Hathaway  BRK-B      10-K       2025.0
SEC_013    extractive Berkshire Hathaway  BRK-B      10-Q       2023.0
SEC_014    extractive  Johnson & Johnson    JNJ      10-Q       2025.0
SEC_015    extractive  Johnson & Johnson    JNJ      10-K       2025.0
SEC_016    extractive UnitedHealth Group    UNH      10-Q       2024.0
SEC_017    extractive UnitedHealth Group    UN

In [6]:
# ---------------------------------------------------------------------------
# SEC corpus loading via shared_retriever
# ---------------------------------------------------------------------------

print("Initializing CorpusIndex from shared_retriever.....")
global_index = initialize_corpus(
    chunks_jsonl=CONFIG["sec_chunks_path"],
    chroma_db_path=CONFIG["chroma_db_path"],
)
print(f"✓ CorpusIndex initialized: {global_index.df.shape[0]:,} chunks")
print(f"✓ Adjacent chunk expansion: ENABLED")

Initializing CorpusIndex from shared_retriever.....
Loading chunks from ..\sec_rag_team_share\sec_data\chunks\sec_chunks.jsonl...
Loaded 13275 chunks
Building BM25 index...
Connecting to Chroma DB at ..\sec_rag_team_share\chroma_db...
Chroma collection: 588 vectors
CorpusIndex ready.
✓ CorpusIndex initialized: 13,275 chunks
✓ Adjacent chunk expansion: ENABLED


In [7]:
# NOTE: Dense model and reranker are now loaded globally in shared_retriever.py
# This ensures all frameworks use the same model instances for reproducibility
print("✓ Dense embedder and reranker loaded via shared_retriever")

✓ Dense embedder and reranker loaded via shared_retriever


In [8]:
# NOTE: RetrievedChunk and CorpusIndex are now defined in shared_retriever.py and imported above.
# This ensures all frameworks (LlamaIndex, CrewAI, LangGraph) use identical retrieval logic,
# including BM25+Dense+RRF+Adjacent Expansion+CrossEncoder, eliminating code duplication.

from shared_retriever import RetrievedChunk as SharedRetrievedChunk

# Define RetrievedChunk locally (same schema as shared_retriever)
@dataclass
class RetrievedChunk:
    doc_name: str
    company: str
    ticker: str
    form_type: str
    filing_year: int
    page_num: int
    chunk_id: str
    raw_chunk: str
    contextual_chunk: str
    score: float
    source: str


# Adapter: shared_retriever Dict → LangGraph-compatible RetrievedChunk
def dict_to_retrieved_chunk(d: Dict[str, Any]) -> RetrievedChunk:
    """Bridge shared_retriever Dict output to LangGraph RetrievedChunk format."""
    meta = d.get('metadata', {})
    return RetrievedChunk(
        doc_name=meta.get('doc_name', ''),
        company=meta.get('company', ''),
        ticker=meta.get('ticker', ''),
        form_type=meta.get('form_type', ''),
        filing_year=int(meta.get('filing_year', 0)),
        page_num=int(meta.get('page_num', 0)),
        chunk_id=d.get('chunk_id', ''),
        raw_chunk=d.get('raw_text', ''),
        contextual_chunk=d.get('text', ''),
        score=float(d.get('score', 0.0)),
        source=d.get('source', ''),
    )


def _rrf_score(rank: int, k: int = None) -> float:
    """Reciprocal Rank Fusion scoring (used for ranking operations)."""
    k = CONFIG["rrf_k"] if k is None else k
    return 1.0 / (k + rank + 1)


print('✓ RetrievedChunk and helpers imported from shared_retriever')

✓ RetrievedChunk and helpers imported from shared_retriever


In [9]:
# Import dense_model and reranker from shared_retriever module
import shared_retriever as shared_retriever_module
dense_model = shared_retriever_module._dense_model
reranker = shared_retriever_module._reranker


def make_llm(model_name: str, temperature: float, provider: str):
    if provider == "groq":
        if not GROQ_API_KEY:
            raise ValueError("GROQ_API_KEY is not set.")
        return ChatGroq(model=model_name, temperature=temperature, groq_api_key=GROQ_API_KEY)
    if provider == "gemini":
        if not GEMINI_API_KEY:
            raise ValueError("GEMINI_API_KEY is not set.")
        return ChatGoogleGenerativeAI(model=model_name, temperature=temperature, google_api_key=GEMINI_API_KEY)
    raise ValueError(f"Unknown provider: {provider!r}")


def resolve_role_temperature(role: str, task_name: str = None) -> float:
    if task_name is not None:
        key = f"temperature_{task_name}"
        if key in CONFIG:
            return CONFIG[key]
    return CONFIG[f"temperature_{role}"] if f"temperature_{role}" in CONFIG else 0.0


def get_llm_candidates(role: str, task_name: str = None):
    temperature = resolve_role_temperature(role, task_name)
    candidates = []
    for provider in get_provider_order():
        for model_name in resolve_fallback_model_names(role, provider=provider):
            try:
                llm = make_llm(model_name, temperature, provider=provider)
            except Exception as exc:
                print(f"  [WARN] Skipping {provider}:{model_name} because client setup failed: {exc}")
                continue
            candidates.append({
                "provider": provider,
                "model_name": model_name,
                "llm": llm,
            })
    return candidates


_preferred_models_by_task: Dict[str, str] = {}
_disabled_model_keys: Set[str] = set()


def disable_model_for_session(provider: str, model_name: str) -> None:
    _disabled_model_keys.add(f"{provider}::{model_name}")


def order_model_candidates(role: str, task_name: str = None):
    candidates = get_llm_candidates(role, task_name)
    preference_key = task_name or role
    preferred = _preferred_models_by_task.get(preference_key)
    if not preferred:
        return candidates
    candidate_keys = [f"{candidate['provider']}::{candidate['model_name']}" for candidate in candidates]
    if preferred not in candidate_keys:
        return candidates
    preferred_idx = candidate_keys.index(preferred)
    return candidates[preferred_idx:]

In [10]:
# ---------------------------------------------------------------------------
# Pydantic output schemas
# ---------------------------------------------------------------------------

def _coerce_bool(v) -> bool:
    if isinstance(v, str):
        return v.strip().lower() in ("true", "1", "yes")
    return bool(v)

def _coerce_float(v) -> float:
    if isinstance(v, str):
        text = v.strip()
        if "/" in text:
            try:
                num, denom = text.split("/", 1)
                return float(num.strip()) / float(denom.strip())
            except Exception:
                pass
        return float(text)
    return float(v)


# ── Proposal 1: Query Decomposition ─────────────────────────────────────────

class SubQuery(BaseModel):
    query: Optional[str] = Field(default=None, description="Search-optimized sub-query text for retrieval")
    ticker: Optional[str] = Field(
        default=None,
        description="Company ticker (e.g. AAPL, MSFT) if this sub-query is specific to one company. "
                    "None for cross-company sub-queries."
    )
    filing_year: Optional[int] = Field(
        default=None,
        description="Specific fiscal year (e.g. 2024) if this sub-query is year-specific. None otherwise."
    )
    form_type: Optional[str] = Field(
        default=None,
        description="10-K or 10-Q if the sub-query is specific to one form type. None otherwise."
    )

    @field_validator("query", mode="before")
    @classmethod
    def normalize_query(cls, v):
        if isinstance(v, dict):
            parts = []
            fields = v.get("fields")
            if isinstance(fields, list) and fields:
                parts.append(" ".join(str(item) for item in fields))
            for key in ("query", "query_field", "table", "metric", "line_item", "fiscal_year"):
                value = v.get(key)
                if value is not None and str(value).strip():
                    parts.append(str(value).strip())
            return " | ".join(dict.fromkeys(parts)) if parts else None
        return v


class PlannerOutput(BaseModel):
    needs_decomposition: bool = Field(
        description="True if the question compares multiple time periods, multiple companies, "
                    "or requires data from multiple distinct filings. "
                    "False for single-period, single-company questions."
    )
    sub_queries: List[SubQuery] = Field(
        description="If needs_decomposition=False: exactly one sub-query with the rewritten question. "
                    "If needs_decomposition=True: 2–3 focused sub-queries, one per time period or company."
    )

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "sub_queries" not in normalized:
            for alias in ("queries", "sub_query", "query"):
                if alias in normalized:
                    value = normalized[alias]
                    if isinstance(value, list):
                        normalized["sub_queries"] = value
                    else:
                        normalized["sub_queries"] = [value]
                    break
        if "needs_decomposition" not in normalized:
            sub_queries = normalized.get("sub_queries", [])
            normalized["needs_decomposition"] = isinstance(sub_queries, list) and len(sub_queries) > 1
        return normalized

    @field_validator("needs_decomposition", mode="before")
    @classmethod
    def coerce_bool(cls, v): return _coerce_bool(v)


# ── Standard schemas ─────────────────────────────────────────────────────────

class QueryRewriteOutput(BaseModel):
    rewritten_query: str = Field(description="Concise rewritten query optimized for retrieval")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if isinstance(data, str):
            return {"rewritten_query": data}
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "rewritten_query" not in normalized:
            for alias in ("query", "rewritten", "search_query"):
                value = normalized.get(alias)
                if value is not None and str(value).strip():
                    normalized["rewritten_query"] = str(value).strip()
                    break
        if "rewritten_query" not in normalized and len(normalized) == 1:
            only_key, only_value = next(iter(normalized.items()))
            parts = [str(only_key).strip()]
            if isinstance(only_value, dict):
                for alias in ("company", "ticker", "source", "form_type", "filingDate", "filing_year", "fiscal_year", "value"):
                    value = only_value.get(alias)
                    if value is not None and str(value).strip():
                        parts.append(str(value).strip())
            elif only_value is not None and str(only_value).strip():
                parts.append(str(only_value).strip())
            normalized["rewritten_query"] = " ".join(dict.fromkeys(part for part in parts if part))
        if "rewritten_query" not in normalized:
            parts = []
            for alias in ("company", "ticker", "form_type", "filingDate", "filing_year", "fiscal_year"):
                value = normalized.get(alias)
                if value is not None and str(value).strip():
                    parts.append(str(value).strip())
            for key in ("metric", "line_item", "statement", "statement_type", "section_title"):
                value = normalized.get(key)
                if value is not None and str(value).strip():
                    parts.append(str(value).strip())
            financial_statements = normalized.get("financialStatements")
            if isinstance(financial_statements, list):
                for item in financial_statements[:2]:
                    if isinstance(item, dict):
                        stmt_type = item.get("type")
                        if stmt_type:
                            parts.append(str(stmt_type).strip())
                        data_block = item.get("data")
                        if isinstance(data_block, dict):
                            parts.extend(str(k).strip() for k in list(data_block.keys())[:3] if str(k).strip())
            if parts:
                normalized["rewritten_query"] = " ".join(dict.fromkeys(parts))
        return normalized


class MultiQueryOutput(BaseModel):
    queries: List[str] = Field(description="Alternative retrieval queries")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if isinstance(data, list):
            return {"queries": data}
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "queries" not in normalized:
            for alias in ("variants", "alternative_queries", "multi_queries"):
                if alias in normalized:
                    normalized["queries"] = normalized[alias]
                    break
        queries = normalized.get("queries")
        if isinstance(queries, str):
            normalized["queries"] = [queries]
        elif isinstance(queries, list):
            cleaned = []
            for item in queries:
                if isinstance(item, str):
                    value = item.strip()
                elif isinstance(item, dict):
                    value = (
                        item.get("query")
                        or item.get("text")
                        or item.get("rewritten_query")
                        or item.get("search_query")
                    )
                    value = str(value).strip() if value is not None else ""
                else:
                    value = str(item).strip()
                if value:
                    cleaned.append(value)
            normalized["queries"] = cleaned
        return normalized


class ContextEvalOutput(BaseModel):
    relevant: bool = Field(description="Whether the retrieved context is relevant and sufficient")
    reason: str = Field(description="Short reason")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "relevant" not in normalized and "is_relevant" in normalized:
            normalized["relevant"] = normalized["is_relevant"]
        answer_present = "answer" in normalized
        answer_value = normalized.get("answer")
        answer_text = ""
        if answer_present and answer_value is not None:
            answer_text = str(answer_value).strip().lower()
        reason_text = str(normalized.get("reason", "")).strip().lower()
        combined_text = " ".join(part for part in (answer_text, reason_text) if part)
        if "relevant" not in normalized and answer_present and answer_value is None:
            normalized["relevant"] = False
        if "relevant" not in normalized and "answer" in normalized:
            negative_markers = (
                "not available",
                "n/a",
                "not found",
                "insufficient data",
                "insufficient information",
                "not enough information",
                "cannot determine",
                "unable to determine",
                "cannot answer",
                "does not contain",
                "doesn't contain",
                "cannot infer",
                "unable to infer",
                "missing from the context",
                "not in the context",
                "not relevant",
            )
            if any(marker in combined_text for marker in negative_markers):
                normalized["relevant"] = False
        if "relevant" not in normalized and reason_text:
            if any(marker in reason_text for marker in ("does not provide", "does not contain", "not provide", "not contain", "cannot determine", "not enough information", "insufficient information", "not relevant")):
                normalized["relevant"] = False
        if "relevant" not in normalized:
            signal_keys = {
                k for k in normalized.keys()
                if k not in {"reason", "is_sufficient", "answer", "is_relevant"}
            }
            if signal_keys:
                normalized["relevant"] = True
        if "reason" not in normalized:
            if "is_sufficient" in normalized:
                normalized["reason"] = f"Model returned is_sufficient={normalized['is_sufficient']}"
            elif "answer" in normalized:
                normalized["reason"] = f"Model returned answer={normalized['answer']}"
            else:
                normalized["reason"] = "No reason provided"
        return normalized

    @field_validator("relevant", mode="before")
    @classmethod
    def coerce_bool(cls, v): return _coerce_bool(v)


# ── Proposal 2: Refined Critic ───────────────────────────────────────────────

class CriticOutput(BaseModel):
    decision: Literal["accept", "repair", "insufficient_data"] = Field(
        description=(
            "accept: the answer is grounded in the context (even if partial). "
            "repair: the answer has a fixable error — wrong period, wrong metric, wrong units, "
            "or directly contradicts the context. The data IS present but the answer is wrong. "
            "insufficient_data: the required financial data is completely absent from ALL retrieved "
            "chunks — not merely incomplete. Never use this just because the question is complex "
            "or the answer is approximate."
        )
    )
    reason: str = Field(description="Short critique")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        for alias in ("verdict", "label", "result", "status"):
            if "decision" not in normalized and alias in normalized:
                normalized["decision"] = normalized[alias]
                break
        if "decision" not in normalized:
            if _coerce_bool(normalized.get("repair")) is True:
                normalized["decision"] = "repair"
            elif _coerce_bool(normalized.get("accept")) is True:
                normalized["decision"] = "accept"
            elif _coerce_bool(normalized.get("insufficient_data")) is True:
                normalized["decision"] = "insufficient_data"
            elif _coerce_bool(normalized.get("refuse")) is True:
                normalized["decision"] = "insufficient_data"
            elif "answer" in normalized:
                normalized["decision"] = "accept"
        for alias in ("rationale", "explanation", "feedback", "critique", "comment"):
            if "reason" not in normalized and alias in normalized:
                normalized["reason"] = normalized[alias]
                break
        if isinstance(normalized.get("reason"), dict):
            normalized["reason"] = "; ".join(f"{k}={v}" for k, v in normalized["reason"].items())
        elif normalized.get("reason") is not None:
            normalized["reason"] = str(normalized["reason"]).strip()
        if "reason" not in normalized:
            if "answer" in normalized:
                normalized["reason"] = f"Model returned answer-shaped critique: {normalized['answer']}"
            else:
                normalized["reason"] = "No reason provided"
        return normalized

    @field_validator("decision", mode="before")
    @classmethod
    def normalize_decision(cls, v):
        if isinstance(v, str):
            raw = v.strip().lower().replace("-", "_").replace(" ", "_")
            mapping = {
                "accept": "accept",
                "accepted": "accept",
                "approve": "accept",
                "approved": "accept",
                "pass": "accept",
                "grounded": "accept",
                "supported": "accept",
                "repair": "repair",
                "revise": "repair",
                "revised": "repair",
                "fix": "repair",
                "correct": "repair",
                "retry": "repair",
                "insufficient_data": "insufficient_data",
                "insufficient": "insufficient_data",
                "missing_data": "insufficient_data",
                "no_data": "insufficient_data",
                "not_enough_data": "insufficient_data",
                "cannot_answer": "insufficient_data",
                "unanswerable": "insufficient_data",
            }
            return mapping.get(raw, raw)
        return v


class RepairOutput(BaseModel):
    decision: Literal["accept", "warn", "refuse"] = Field(
        description=(
            "Final decision after repair. Use 'accept' when you have a revised answer — "
            "even partial. Use 'warn' only for genuine ambiguity. "
            "Use 'refuse' ONLY if the question is entirely out of scope."
        )
    )
    revised_answer: str = Field(description="The final revised answer.")
    needs_new_retrieval: bool = Field(
        description="true or false — whether fresh retrieval would help. Must be a boolean, not a string."
    )
    reason: str = Field(description="Short rationale")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "data" in normalized and isinstance(normalized["data"], dict):
            for key, value in normalized["data"].items():
                normalized.setdefault(key, value)
        for alias in ("answer", "final_answer", "response"):
            if "revised_answer" not in normalized and alias in normalized:
                normalized["revised_answer"] = normalized[alias]
                break
        if "revised_answer" not in normalized:
            payload_keys = [
                k for k in normalized.keys()
                if k not in {"decision", "reason", "needs_new_retrieval", "data"}
            ]
            if payload_keys:
                normalized["revised_answer"] = "; ".join(
                    f"{k}={normalized[k]}" for k in payload_keys
                )
        for alias in ("rationale", "explanation", "feedback", "comment", "context"):
            if "reason" not in normalized and alias in normalized:
                normalized["reason"] = normalized[alias]
                break
        if isinstance(normalized.get("reason"), (dict, list)):
            normalized["reason"] = json.dumps(normalized["reason"], ensure_ascii=False)
        elif normalized.get("reason") is not None:
            normalized["reason"] = str(normalized["reason"]).strip()
        if "needs_new_retrieval" not in normalized:
            normalized["needs_new_retrieval"] = False
        if "reason" not in normalized:
            if "revised_answer" in normalized:
                normalized["reason"] = "Model returned a revised answer"
            else:
                normalized["reason"] = "No reason provided"
        return normalized
    @field_validator("decision", mode="before")
    @classmethod
    def normalize_decision(cls, v):
        if isinstance(v, str):
            raw = v.strip().lower().replace("-", "_").replace(" ", "_")
            mapping = {
                "accept": "accept",
                "accepted": "accept",
                "warn": "warn",
                "warning": "warn",
                "refuse": "refuse",
                "refusal": "refuse",
                "reject": "refuse",
            }
            return mapping.get(raw, raw)
        return v

    @field_validator("reason", mode="before")
    @classmethod
    def normalize_reason(cls, v):
        if isinstance(v, (dict, list)):
            return json.dumps(v, ensure_ascii=False)
        if v is None:
            return "No reason provided"
        return str(v).strip()

    @field_validator("needs_new_retrieval", mode="before")
    @classmethod
    def coerce_bool(cls, v): return _coerce_bool(v)


class JudgeOutput(BaseModel):
    score: float = Field(description="Overall correctness score — decimal number between 0.0 and 1.0. Use decimals only, never fractions like 2/3.")
    claims_covered: float = Field(
        description="Fraction of key facts covered — decimal number between 0.0 and 1.0. Use decimals only, never fractions like 2/3."
    )
    reason: str = Field(description="Short explanation")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        for alias in ("overall_score", "correctness_score", "faithfulness_score", "grounding_score", "grounded_score", "rating", "grade"):
            if "score" not in normalized and alias in normalized:
                normalized["score"] = normalized[alias]
                break
        if "score" not in normalized and "hallucination" in normalized:
            try:
                hallucination = _coerce_float(normalized["hallucination"])
                normalized["score"] = max(0.0, min(1.0, 1.0 - hallucination))
            except Exception:
                pass
        for alias in ("coverage", "coverage_score", "supported_fraction", "support_rate", "claim_coverage"):
            if "claims_covered" not in normalized and alias in normalized:
                normalized["claims_covered"] = normalized[alias]
                break
        for alias in ("rationale", "explanation", "feedback", "comment"):
            if "reason" not in normalized and alias in normalized:
                normalized["reason"] = normalized[alias]
                break
        if isinstance(normalized.get("reason"), dict):
            normalized["reason"] = "; ".join(f"{k}={v}" for k, v in normalized["reason"].items())
        if "claims_covered" not in normalized and "score" in normalized:
            normalized["claims_covered"] = normalized["score"]
        if "reason" not in normalized:
            normalized["reason"] = "No reason provided"
        return normalized

    @field_validator("score", "claims_covered", mode="before")
    @classmethod
    def coerce_float(cls, v): return _coerce_float(v)

In [11]:
# ---------------------------------------------------------------------------
# Prompt templates
# ---------------------------------------------------------------------------

def _question_tokens(text: str) -> set:
    return set(re.findall(r"[A-Za-z0-9]+", str(text).lower()))


def select_few_shot_examples(question: str, max_examples: int = None) -> str:
    if not CONFIG.get("use_few_shot_examples", False) or "train_df" not in globals() or train_df.empty:
        return ""
    top_rows = train_df.sort_values("id").reset_index(drop=True).itertuples(index=False)
    rendered = []
    for idx, row in enumerate(top_rows, start=1):
        rendered.append(
            f"Example {idx}\n"
            f"Question: {getattr(row, 'question', '')}\n"
            f"Reference answer: {getattr(row, 'reference_answer', '')}\n"
            f"Evidence: {getattr(row, 'evidence_text', '')}"
        )
    return "\n\n".join(rendered)


# ── Proposal 1: Planner ───────────────────────────────────────────────────────
planner_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a financial research planner working with SEC filings from these companies:\n"
        "AAPL (Apple), BAC (Bank of America), BRK (Berkshire Hathaway), COST (Costco), "
        "CVX (Chevron), JNJ (Johnson & Johnson), JPM (JPMorgan Chase), MSFT (Microsoft), "
        "NVDA (NVIDIA), UNH (UnitedHealth), WMT (Walmart), XOM (ExxonMobil).\n\n"
        "Decide whether the question requires data from multiple distinct filings, then produce sub-queries.\n\n"
        "Decompose (needs_decomposition=True) when the question:\n"
        "  • Explicitly compares two different fiscal years (e.g. 2023 vs 2024)\n"
        "  • Compares two different companies\n\n"
        "Do NOT decompose single-period, single-company questions.\n\n"
        "For each sub-query set ticker if company-specific, filing_year if year-specific.\n"
        "Every sub-query object must include a non-empty query field.\n"
        "filing_year = the calendar year the 10-K or 10-Q was filed "
        "(e.g. Apple FY2024 10-K was filed in November 2024, so filing_year=2024). "
        "Return a valid JSON object only that matches the requested schema."
    ),
    ("human", "Question: {question}"),
])

context_eval_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You judge whether retrieved context is relevant and sufficient to answer a financial question. "
        "Mark as not relevant only if the context is clearly from the wrong company/period, or completely empty. "
        "Partial tables or incomplete passages still count as relevant if they contain the right metric. "
        "Return a valid JSON object only that matches the requested schema. "
        "For boolean fields, use true/false. For numeric fields, use decimals like 0.67, never fractions like 2/3. "
        "For explanation fields, return a plain string, not an object.",
    ),
    ("human", "Question:\n{question}\n\nRetrieved context:\n{context}"),
])

generator_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a financial analyst answering questions using only the retrieved filing context. "
        "Be precise with numbers — include units (millions, billions, %), fiscal periods, and line item names. "
        "If the context does not contain the answer, say so explicitly rather than estimating. "
        "Use the examples only as answer-format/style guidance, never as facts for the current question.",
    ),
    ("human", "Examples:\n{few_shot_examples}\n\nQuestion:\n{question}\n\nRetrieved context:\n{context}"),
])

# ── Proposal 2: Updated critic prompt ────────────────────────────────────────
critic_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a critic for a financial RAG system. Evaluate the answer and choose one decision:\n\n"
        "  accept          — the answer is numerically grounded in the context (even if partial).\n"
        "  repair          — the answer has a fixable error: wrong period, wrong metric, wrong units, "
        "or contradicts the context. The data IS present but the answer used it incorrectly.\n"
        "  insufficient_data — the required financial data is completely absent from ALL retrieved "
        "chunks. Only use this when no amount of repair can help — the filing simply does not "
        "contain the information. Never use this just because the answer is approximate. "
        "Return a valid JSON object only that matches the requested schema. "
        "For boolean fields, use true/false. For numeric fields, use decimals like 0.67, never fractions like 2/3. "
        "For explanation fields, return a plain string, not an object.",
    ),
    ("human", "Question:\n{question}\n\nRetrieved context:\n{context}\n\nCurrent answer:\n{answer}"),
])

repair_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You repair a financial RAG answer after critique. Your default action is to ACCEPT with a revised answer. "
        "Pay close attention to: correct fiscal year/quarter, correct units (millions vs billions), "
        "correct sign, and the right line item name. "
        "Set decision='accept' whenever you can produce any useful revised answer — even partial. "
        "Set decision='warn' only if the context is genuinely ambiguous. "
        "Set decision='refuse' ONLY if the question is entirely off-topic for all retrieved filings. "
        "Set needs_new_retrieval=true only if critical data is completely absent from the context. "
        "Return a valid JSON object only that matches the requested schema."
    ),
    (
        "human",
        "Question:\n{question}\n\nRetrieved context:\n{context}\n\n"
        "Original answer:\n{answer}\n\nCritic feedback:\n{critique}",
    ),
])

judge_correctness_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. "
        "1 = correct value, correct units, correct period. "
        "0 = wrong number, wrong company, wrong period, or completely missing. "
        "Give partial credit for answers close but with unit errors. "
        "Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate. "
        "Return a valid JSON object only that matches the requested schema. "
        "For boolean fields, use true/false. For numeric fields, use decimals like 0.67, never fractions like 2/3. "
        "For explanation fields, return a plain string, not an object.",
    ),
    (
        "human",
        "Question:\n{question}\n\nReference answer:\n{reference_answer}\n\nCandidate answer:\n{candidate_answer}",
    ),
])

judge_faithfulness_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Score how well the candidate answer is grounded in the retrieved filing context from 0 to 1. "
        "1 = every number and claim is directly supported by the context. "
        "0 = answer contains numbers or claims not present in the context (hallucinated). "
        "Also set claims_covered: fraction of claims in the candidate supported by the context. "
        "Return a valid JSON object only that matches the requested schema. "
        "For boolean fields, use true/false. For numeric fields, use decimals like 0.67, never fractions like 2/3. "
        "For explanation fields, return a plain string, not an object.",
    ),
    (
        "human",
        "Question:\n{question}\n\nRetrieved context:\n{context}\n\nCandidate answer:\n{candidate_answer}",
    ),
])

rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a search query optimizer for SEC filings (10-K, 10-Q). "
        "Rewrite the user's question as a concise retrieval query that maximizes recall of relevant "
        "financial document chunks. Use precise financial terminology such as net revenue, operating income, "
        "or diluted EPS. Keep it under 25 words. Return a valid JSON object only that matches the requested schema.",
    ),
    ("human", "Question: {question}"),
])

multi_query_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Generate {n} alternative retrieval queries for a question about SEC filings. "
        "Each variant should use different financial terminology or focus on a different aspect of the question "
        "to maximise retrieval coverage. Return a valid JSON object only that matches the requested schema.",
    ),
    ("human", "Question: {question}"),
])

generator_citations_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a financial analyst answering questions using only the retrieved filing context. "
        "Be precise with numbers and include units, fiscal periods, and line item names. "
        "Cite sources using [n] notation after each claim. "
        "If the context does not contain enough information, say so explicitly instead of guessing.",
    ),
    ("human", "Question:\n{question}\n\nRetrieved context (cite [n] for each claim):\n{context}"),
])


In [12]:
# ---------------------------------------------------------------------------
# Rate limiter + safe LLM invocation helpers
# ---------------------------------------------------------------------------

class RateLimiter:
    def __init__(self, max_rpm: int):
        self.max_rpm = max_rpm
        self.window = 60.0
        self.timestamps: deque = deque()
        self._lock = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.time()
            while self.timestamps and now - self.timestamps[0] >= self.window:
                self.timestamps.popleft()
            if len(self.timestamps) >= self.max_rpm:
                sleep_for = self.window - (now - self.timestamps[0])
                if sleep_for > 0:
                    print(f"  [RateLimit] {self.max_rpm} RPM cap — waiting {sleep_for:.1f}s")
                    time.sleep(sleep_for)
            self.timestamps.append(time.time())


_rate_limiters = {
    provider: RateLimiter(max_rpm=rpm)
    for provider, rpm in CONFIG["provider_rpm"].items()
}


def get_rate_limiter(provider: str) -> RateLimiter:
    return _rate_limiters.get(provider, RateLimiter(max_rpm=CONFIG["max_rpm"]))


def is_rate_limit_error(exc: Exception) -> bool:
    message = str(exc).lower()
    return "rate limit" in message or "rate_limit" in message or "429" in message


def is_model_exhausted_error(exc: Exception) -> bool:
    message = str(exc).lower()
    exhaustion_markers = (
        "tokens per day",
        "tpd",
        "quota exceeded",
        "resource_exhausted",
        "limit 500000",
        "generate_content_free_tier_requests",
        "generatecontentinputtokens",
    )
    return any(marker in message for marker in exhaustion_markers)


def safe_invoke_structured(
    role: str,
    schema_class,
    prompt,
    variables: Dict[str, Any],
    fallback: BaseModel,
    task_name: str = None,
) -> BaseModel:
    # Use json_mode to bypass Groq's strict tool-call schema validation.
    # The model outputs raw JSON which Pydantic then parses — field_validators
    # handle any string→bool/float coercion the model produces.
    model_candidates = order_model_candidates(role, task_name)
    if not model_candidates:
        print(f"  [WARN] No available model candidates remain for {task_name or role}; returning fallback text.")
        return fallback_text
    if not model_candidates:
        print(f"  [WARN] No available model candidates remain for {task_name or role}; using fallback.")
        return fallback
    preference_key = task_name or role
    max_attempts = CONFIG["llm_max_retries"]
    for model_idx, candidate in enumerate(model_candidates):
        provider = candidate["provider"]
        model_name = candidate["model_name"]
        llm = candidate["llm"]
        for attempt in range(max_attempts):
            try:
                get_rate_limiter(provider).wait()
                structured = llm.with_structured_output(schema_class, method="json_mode")
                result = (prompt | structured).invoke(variables)
                _preferred_models_by_task[preference_key] = f"{provider}::{model_name}"
                return result
            except Exception as e:
                print(
                    f"  [WARN] {schema_class.__name__} attempt {attempt+1}/{max_attempts} "
                    f"on {provider}:{model_name} failed: {e}"
                )
                if is_model_exhausted_error(e):
                    disable_model_for_session(provider, model_name)
                    print(f"  [WARN] Disabling {provider}:{model_name} for the rest of this session.")
                should_switch = is_rate_limit_error(e) and model_idx < len(model_candidates) - 1
                if should_switch:
                    next_candidate = model_candidates[model_idx + 1]
                    next_key = f"{next_candidate['provider']}::{next_candidate['model_name']}"
                    _preferred_models_by_task[preference_key] = next_key
                    print(
                        f"  [WARN] Switching {role} model from {provider}:{model_name} "
                        f"to {next_candidate['provider']}:{next_candidate['model_name']} after rate limit."
                    )
                    break
                if attempt == max_attempts - 1:
                    if model_idx == len(model_candidates) - 1:
                        print(f"  [WARN] All retries exhausted for {schema_class.__name__}, using fallback.")
                        return fallback
                    continue
                delay = CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt)
                time.sleep(delay)
    return fallback


def safe_invoke_llm(
    role: str,
    prompt,
    variables: Dict[str, Any],
    fallback_text: str = "",
    task_name: str = None,
) -> str:
    model_candidates = order_model_candidates(role, task_name)
    preference_key = task_name or role
    max_attempts = CONFIG["llm_max_retries"]
    for model_idx, candidate in enumerate(model_candidates):
        provider = candidate["provider"]
        model_name = candidate["model_name"]
        llm = candidate["llm"]
        chain = prompt | llm
        for attempt in range(max_attempts):
            try:
                get_rate_limiter(provider).wait()
                result = chain.invoke(variables).content
                _preferred_models_by_task[preference_key] = f"{provider}::{model_name}"
                return result
            except Exception as e:
                print(f"  [WARN] LLM call attempt {attempt+1}/{max_attempts} on {provider}:{model_name} failed: {e}")
                if is_model_exhausted_error(e):
                    disable_model_for_session(provider, model_name)
                    print(f"  [WARN] Disabling {provider}:{model_name} for the rest of this session.")
                should_switch = is_rate_limit_error(e) and model_idx < len(model_candidates) - 1
                if should_switch:
                    next_candidate = model_candidates[model_idx + 1]
                    next_key = f"{next_candidate['provider']}::{next_candidate['model_name']}"
                    _preferred_models_by_task[preference_key] = next_key
                    print(
                        f"  [WARN] Switching {role} model from {provider}:{model_name} "
                        f"to {next_candidate['provider']}:{next_candidate['model_name']} after rate limit."
                    )
                    break
                if attempt == max_attempts - 1:
                    if model_idx == len(model_candidates) - 1:
                        return fallback_text
                    continue
                delay = CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt)
                time.sleep(delay)
    return fallback_text


def format_retrieved_context(
    chunks: List[RetrievedChunk],
    max_chunks: int = None,
    max_chars: int = None,
) -> str:
    selected = list(chunks[: max_chunks or len(chunks)])
    parts = []
    total_chars = 0
    for i, c in enumerate(selected, start=1):
        parts.append(
            f"[Doc {i}] Company: {c.company} | Filing: {c.doc_name}\n"
            f"Content: {c.raw_chunk}"
        )
        total_chars = sum(len(p) for p in parts) + max(0, 2 * (len(parts) - 1))
        if max_chars is not None and total_chars >= max_chars:
            joined = "\n\n".join(parts)
            return joined[:max_chars]
    return "\n\n".join(parts)


def extract_retrieved_doc_names(chunks: List[RetrievedChunk]) -> List[str]:
    return list(dict.fromkeys([c.doc_name for c in chunks]))


def cleanup_planner_query(query: str, ticker: str = None, filing_year: int = None, form_type: str = None) -> str:
    cleaned = (query or "").strip()
    if not cleaned:
        cleaned = "financial metric from SEC filing"
    upper = cleaned.upper()
    if upper.startswith("SELECT ") and " FROM " in upper:
        cleaned = cleaned.replace("SELECT", "").replace("FROM filings", "").strip()
    for token in ("WHERE", "FROM", "AND", "=", "'", '"'):
        cleaned = cleaned.replace(token, " ")
    cleaned = cleaned.replace("filing year", "filing_year")
    cleaned = cleaned.replace("|", " ")
    cleaned = cleaned.replace("_", " ")
    cleaned = cleaned.replace("company", "")
    cleaned = cleaned.replace("ticker", "")
    cleaned = " ".join(cleaned.split())
    parts = [cleaned]
    if ticker and ticker not in cleaned:
        parts.insert(0, ticker)
    if filing_year and str(filing_year) not in cleaned:
        parts.append(str(filing_year))
    if form_type and form_type not in cleaned:
        parts.append(form_type)
    return " ".join(part for part in parts if part).strip()


KNOWN_TICKERS = {
    "AAPL", "MSFT", "NVDA", "JPM", "BAC", "BRK.B", "BRK",
    "JNJ", "UNH", "XOM", "CVX", "WMT", "COST",
}

COMPANY_TO_TICKER = {
    "apple": "AAPL",
    "microsoft": "MSFT",
    "nvidia": "NVDA",
    "jpmorgan": "JPM",
    "jp morgan": "JPM",
    "bank of america": "BAC",
    "berkshire": "BRK.B",
    "johnson": "JNJ",
    "unitedhealth": "UNH",
    "exxon": "XOM",
    "exxonmobil": "XOM",
    "chevron": "CVX",
    "walmart": "WMT",
    "costco": "COST",
}


def extract_metadata(query: str) -> Dict[str, Any]:
    q_upper = query.upper()
    q_lower = query.lower()

    ticker = None
    for candidate in KNOWN_TICKERS:
        if re.search(r"\b" + re.escape(candidate) + r"\b", q_upper):
            ticker = candidate
            break
    if ticker is None:
        for company_name, candidate in COMPANY_TO_TICKER.items():
            if company_name in q_lower:
                ticker = candidate
                break

    year_match = re.search(r"\b(202[0-9])\b", query)
    filing_year = int(year_match.group(1)) if year_match else None

    form_type = None
    if re.search(r"\b10-K\b|\bannual report\b|\bannual filing\b", query, re.IGNORECASE):
        form_type = "10-K"
    elif re.search(r"\b10-Q\b|\bquarterly report\b|\bquarterly filing\b", query, re.IGNORECASE):
        form_type = "10-Q"

    return {"ticker": ticker, "filing_year": filing_year, "form_type": form_type}


def rewrite_query(query: str) -> str:
    result = safe_invoke_structured(
        "agent",
        QueryRewriteOutput,
        rewrite_prompt,
        {"question": query},
        QueryRewriteOutput(rewritten_query=query),
        task_name="rewrite",
    )
    rewritten = (getattr(result, "rewritten_query", "") or "").strip()
    return rewritten or query


def generate_multi_queries(query: str, n: int = None) -> List[str]:
    n = n or CONFIG["multi_query_n"]
    result = safe_invoke_structured(
        "agent",
        MultiQueryOutput,
        multi_query_prompt,
        {"question": query, "n": n},
        MultiQueryOutput(queries=[]),
        task_name="multi_query",
    )
    deduped = []
    seen = {query.strip().lower()}
    for candidate in getattr(result, "queries", []) or []:
        candidate = str(candidate).strip()
        if not candidate:
            continue
        key = candidate.lower()
        if key in seen:
            continue
        seen.add(key)
        deduped.append(candidate)
        if len(deduped) >= n:
            break
    return deduped


def rrf_merge_retrieved(ranked_lists: List[List[RetrievedChunk]], k: int = None) -> List[RetrievedChunk]:
    k = CONFIG["rrf_k"] if k is None else k
    rrf_scores: Dict[str, float] = {}
    pool: Dict[str, RetrievedChunk] = {}
    for ranked in ranked_lists:
        for rank, chunk in enumerate(ranked):
            rrf_scores[chunk.chunk_id] = rrf_scores.get(chunk.chunk_id, 0.0) + _rrf_score(rank, k)
            if chunk.chunk_id not in pool:
                pool[chunk.chunk_id] = chunk
    return [pool[key] for key in sorted(rrf_scores, key=rrf_scores.__getitem__, reverse=True)]


def rerank_retrieved(candidates: List[RetrievedChunk], query: str, top_k: int = None) -> List[RetrievedChunk]:
    top_k = top_k or CONFIG["rerank_top_k"]
    if not candidates:
        return []
    scores = reranker.predict(
        [(query, chunk.contextual_chunk) for chunk in candidates],
        show_progress_bar=False,
    )
    reranked = []
    for chunk, score in zip(candidates, scores):
        reranked.append(
            RetrievedChunk(
                doc_name=chunk.doc_name,
                company=chunk.company,
                ticker=chunk.ticker,
                form_type=chunk.form_type,
                filing_year=chunk.filing_year,
                page_num=chunk.page_num,
                chunk_id=chunk.chunk_id,
                raw_chunk=chunk.raw_chunk,
                contextual_chunk=chunk.contextual_chunk,
                score=float(score),
                source="hybrid_reranked",
            )
        )
    return sorted(reranked, key=lambda chunk: chunk.score, reverse=True)[:top_k]


def compress_retrieved_chunk(chunk: RetrievedChunk, query: str, top_n: int = None) -> RetrievedChunk:
    top_n = top_n or CONFIG["compress_top_sentences"]
    if top_n is None:
        return chunk

    text = chunk.contextual_chunk
    if "\nContent:" in text:
        header, _, content = text.partition("\nContent:")
        header = header + "\nContent:"
    else:
        header = ""
        content = text

    sentences = re.split(r"(?<=[.!?;])\s+", content.strip())
    sentences = [sentence.strip() for sentence in sentences if len(sentence.strip()) > 25]
    if len(sentences) <= top_n:
        return chunk

    scores = reranker.predict([(query, sentence) for sentence in sentences], show_progress_bar=False)
    top_idx = sorted(sorted(range(len(scores)), key=lambda idx: scores[idx], reverse=True)[:top_n])
    compressed_content = " ".join(sentences[idx] for idx in top_idx)
    contextual_chunk = (header + " " + compressed_content).strip()
    return RetrievedChunk(
        doc_name=chunk.doc_name,
        company=chunk.company,
        ticker=chunk.ticker,
        form_type=chunk.form_type,
        filing_year=chunk.filing_year,
        page_num=chunk.page_num,
        chunk_id=chunk.chunk_id,
        raw_chunk=compressed_content,
        contextual_chunk=contextual_chunk,
        score=chunk.score,
        source=f"{chunk.source}_compressed",
    )


def compress_retrieved_context(chunks: List[RetrievedChunk], query: str) -> List[RetrievedChunk]:
    return [compress_retrieved_chunk(chunk, query) for chunk in chunks]


def format_context_adv(chunks: List[RetrievedChunk], max_chars: int = None) -> str:
    max_chars = max_chars or CONFIG["max_context_chars"]
    parts = []
    for idx, chunk in enumerate(chunks, start=1):
        header = f"[{idx}] {chunk.ticker} {chunk.form_type} {chunk.filing_year} | {chunk.company}"
        parts.append(f"{header}\n{chunk.raw_chunk}")
    return "\n\n---\n\n".join(parts)[:max_chars]


def generate_with_citations(query: str, chunks: List[RetrievedChunk]) -> str:
    context = format_context_adv(chunks)
    return safe_invoke_llm(
        "generator",
        generator_citations_prompt,
        {"question": query, "context": context},
        task_name="generator",
    )


def run_advanced_retrieval(
    index: CorpusIndex,
    query: str,
    meta_override: Dict[str, Any] = None,
    rerank_top_k: int = None,
    verbose: bool = False,
) -> Dict[str, Any]:
    meta = extract_metadata(query)
    for key, value in (meta_override or {}).items():
        if value is not None:
            meta[key] = value

    rewritten = rewrite_query(query)
    variants = generate_multi_queries(rewritten)
    all_queries = [rewritten] + variants

    if verbose:
        print(f"  [Meta]     {meta}")
        print(f"  [Rewrite]  {rewritten}")
        print(f"  [Queries]  {all_queries}")

    bm25_mask = index._bm25_mask(meta.get("ticker"), meta.get("filing_year"), meta.get("form_type"))
    ranked_lists: List[List[RetrievedChunk]] = []
    for variant in all_queries:
        ranked_lists.append(index.bm25_search(variant, CONFIG["bm25_top_k"], bm25_mask))
        ranked_lists.append(
            index.dense_search(
                variant,
                CONFIG["dense_top_k"],
                ticker=meta.get("ticker"),
                filing_year=meta.get("filing_year"),
                form_type=meta.get("form_type"),
            )
        )

    candidates = rrf_merge_retrieved(ranked_lists)
    reranked = rerank_retrieved(candidates, rewritten, top_k=rerank_top_k or CONFIG["rerank_top_k"])
    compressed = compress_retrieved_context(reranked, rewritten)

    if verbose:
        print(f"  [RRF]      {len(candidates)} unique candidates after merge")
        print(f"  [Rerank]   top {len(reranked)} chunks selected")

    return {
        "query": query,
        "rewritten": rewritten,
        "variants": variants,
        "metadata": meta,
        "reranked_chunks": reranked,
        "retrieved_chunks": compressed,
    }


def fails_retrieval_sanity_check(question: str, sub_queries: List[Dict[str, Any]], retrieved: List[RetrievedChunk]) -> bool:
    if not CONFIG.get("enable_retrieval_sanity_check", False):
        return False
    if not retrieved:
        return False
    expected_tickers = {sq.get("ticker") for sq in sub_queries if sq.get("ticker")}
    if not expected_tickers:
        return False
    present_tickers = {chunk.ticker for chunk in retrieved if getattr(chunk, "ticker", None)}
    if not present_tickers:
        return False
    return not expected_tickers.issubset(present_tickers)


def get_active_model_name(role: str, task_name: str = None) -> str:
    candidates = order_model_candidates(role, task_name)
    if not candidates:
        return resolve_model_name(role)
    first = candidates[0]
    if isinstance(first, dict):
        return first["model_name"]
    return first[0]


def get_task_model_snapshot(task_names: List[str]) -> Dict[str, str]:
    snapshot: Dict[str, str] = {}
    for task_name in task_names:
        preferred = _preferred_models_by_task.get(task_name)
        if preferred:
            snapshot[task_name] = preferred
            continue
        candidates = order_model_candidates("agent" if task_name in {"planner", "rewrite", "multi_query", "context_eval", "critic", "repair"} else task_name, task_name)
        if candidates:
            first = candidates[0]
            if isinstance(first, dict):
                snapshot[task_name] = f"{first['provider']}::{first['model_name']}"
            else:
                snapshot[task_name] = first[0]
    return snapshot


def get_model_aware_context_limits(
    role: str,
    task_name: str,
    default_chunks: int,
    default_chars: int,
) -> Tuple[int, int]:
    model_name = get_active_model_name(role, task_name).lower()
    if "llama-3.1-8b-instant" in model_name:
        budgets = {
            "context_eval": (2, 2200),
            "critic": (2, 2500),
            "repair": (2, 2500),
            "judge": (2, 2200),
            "generator": (5, 7000),
        }
        return budgets.get(task_name, (default_chunks, default_chars))
    if "qwen/qwen3-32b" in model_name:
        budgets = {
            "context_eval": (3, 3500),
            "critic": (3, 4000),
            "repair": (3, 4000),
            "judge": (3, 3200),
            "generator": (6, 9000),
        }
        return budgets.get(task_name, (default_chunks, default_chars))
    return default_chunks, default_chars

In [13]:

# ─────────────────────────────────────────────────────────────────────────────
# DRIFT DETECTION + AUTO-INGESTION INFRASTRUCTURE
# ─────────────────────────────────────────────────────────────────────────────
# Detects when users consistently query for data not yet in the corpus
# (e.g. FY2025 filings when the DB only has up to FY2024), then automatically
# scrapes the missing filing from SEC EDGAR and ingests it into ChromaDB.
#
# Graph flow (new branch from critic's insufficient_data path):
#   critic (insufficient_data)
#     → node_drift_detector  [logs miss, checks TRIGGER_THRESHOLD]
#       → node_scrape_and_ingest  [EDGAR → chunk → upsert ChromaDB → rebuild BM25]
#         → hybrid_retriever  [retry with freshly ingested data]
# ─────────────────────────────────────────────────────────────────────────────

import requests
from collections import defaultdict
from datetime import datetime
from pathlib import Path

# ── Ticker → CIK mapping (same 12 companies as the corpus) ──────────────────
TICKER_TO_CIK = {
    "AAPL":  "0000320193", "MSFT":  "0000789019", "NVDA":  "0001045810",
    "JPM":   "0000019617", "BAC":   "0000070858", "BRK-B": "0001067983",
    "JNJ":   "0000200406", "UNH":   "0000731766", "XOM":   "0000034088",
    "CVX":   "0000093410", "WMT":   "0000104169", "COST":  "0000909832",
}

COMPANY_NAMES_MAP = {
    "AAPL":  "Apple Inc.",                         "MSFT":  "Microsoft Corporation",
    "NVDA":  "NVIDIA Corporation",                 "JPM":   "JPMorgan Chase & Co.",
    "BAC":   "Bank of America Corporation",        "BRK-B": "Berkshire Hathaway Inc.",
    "JNJ":   "Johnson & Johnson",                  "UNH":   "UnitedHealth Group Incorporated",
    "XOM":   "Exxon Mobil Corporation",            "CVX":   "Chevron Corporation",
    "WMT":   "Walmart Inc.",                       "COST":  "Costco Wholesale Corporation",
}

# EDGAR requires a descriptive User-Agent with contact info
EDGAR_HEADERS = {"User-Agent": "GenAI-RAG-Project research@example.com"}



def drift_log(message: str, *, verbose_only: bool = False) -> None:
    if verbose_only and not CONFIG.get("verbose_drift_logging", False):
        return
    print(message)


def canonical_drift_key(ticker: Optional[str], filing_year: Optional[str], form_type: Optional[str]) -> tuple:
    ticker_norm = str(ticker).strip().upper() if ticker not in (None, "", "None") else None
    if filing_year in (None, "", "None"):
        year_norm = None
    else:
        try:
            year_norm = str(int(filing_year))
        except Exception:
            year_norm = str(filing_year).strip()
    form_norm = str(form_type or "10-K").strip().upper()
    return (ticker_norm, year_norm, form_norm)


# ─────────────────────────────────────────────────────────────────────────────
# DriftTracker — accumulates corpus-miss signals and decides when to scrape
# ─────────────────────────────────────────────────────────────────────────────

class DriftTracker:
    """
    Logs each 'insufficient_data' critic decision and counts misses per
    (ticker, filing_year, form_type). Once a combo hits TRIGGER_THRESHOLD,
    it schedules a scrape from EDGAR on the next qualifying graph run.
    The log is persisted to JSONL so counts survive session restarts.
    """
    TRIGGER_THRESHOLD = 2
    PERSIST_PATH = Path(CONFIG["sec_chunks_path"]).parent.parent / "drift_log.jsonl"

    def __init__(self):
        self._counts: Dict[tuple, int] = defaultdict(int)
        self._already_ingested: set = set()
        self._failed_scrapes: set = set()
        self._load_existing()

    def _load_existing(self):
        if self.PERSIST_PATH.exists():
            with open(self.PERSIST_PATH, encoding="utf-8") as f:
                for line in f:
                    try:
                        entry = json.loads(line)
                        key = canonical_drift_key(entry.get("ticker"), entry.get("filing_year"), entry.get("form_type"))
                        if entry.get("status") == "ingested":
                            self._already_ingested.add(key)
                        elif entry.get("status") == "failed":
                            self._failed_scrapes.add(key)
                        elif entry.get("status") == "miss":
                            self._counts[key] += 1
                    except Exception:
                        pass

    def log_miss(self, question: str, ticker: Optional[str],
                 filing_year: Optional[str], form_type: Optional[str]) -> None:
        key = canonical_drift_key(ticker, filing_year, form_type)
        if key in self._already_ingested or key in self._failed_scrapes:
            return
        self._counts[key] += 1
        self.PERSIST_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(self.PERSIST_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps({
                "status": "miss", "ticker": key[0], "filing_year": key[1],
                "form_type": key[2], "question": question,
                "timestamp": datetime.utcnow().isoformat(),
            }) + "\n")
        drift_log(f"  [DriftTracker] Miss #{self._counts[key]} for ({key[0]}, {key[1]}, {key[2]})", verbose_only=True)

    def get_pending_scrapes(self) -> List[Tuple[str, str, str]]:
        """Return (ticker, year, form_type) combos that crossed the threshold and are not yet ingested."""
        return [
            (ticker, year, ft)
            for (ticker, year, ft), count in self._counts.items()
            if count >= self.TRIGGER_THRESHOLD
            and (ticker, year, ft) not in self._already_ingested
            and (ticker, year, ft) not in self._failed_scrapes
            and ticker and year
        ]

    def mark_ingested(self, ticker: str, filing_year: str, form_type: str) -> None:
        key = canonical_drift_key(ticker, filing_year, form_type)
        self._already_ingested.add(key)
        with open(self.PERSIST_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps({
                "status": "ingested", "ticker": key[0],
                "filing_year": key[1], "form_type": key[2],
                "timestamp": datetime.utcnow().isoformat(),
            }) + "\n")


    def mark_failed(self, ticker: str, filing_year: str, form_type: str, reason: str) -> None:
        key = canonical_drift_key(ticker, filing_year, form_type)
        self._failed_scrapes.add(key)
        with open(self.PERSIST_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps({
                "status": "failed", "ticker": key[0],
                "filing_year": key[1], "form_type": key[2],
                "reason": reason,
                "timestamp": datetime.utcnow().isoformat(),
            }) + "\n")


drift_tracker = DriftTracker()
drift_log(f"DriftTracker ready | persist path: {drift_tracker.PERSIST_PATH}", verbose_only=True)


# ─────────────────────────────────────────────────────────────────────────────
# SEC EDGAR Scraper
# ─────────────────────────────────────────────────────────────────────────────

def _strip_html(html: str) -> str:
    text = re.sub(r"<[^>]+>", " ", html)
    for ent, repl in [("&nbsp;", " "), ("&amp;", "&"), ("&lt;", "<"), ("&gt;", ">"), ("&#160;", " ")]:
        text = text.replace(ent, repl)
    return re.sub(r"\s{3,}", "\n\n", text).strip()


def fetch_edgar_filings_list(ticker: str, form_type: str = "10-K") -> List[Dict]:
    """Returns recent filings metadata from EDGAR submissions API for a given ticker and form type."""
    cik = TICKER_TO_CIK.get(ticker.upper())
    if not cik:
        raise ValueError(f"Unknown ticker: {ticker}. Add it to TICKER_TO_CIK.")
    resp = requests.get(
        f"https://data.sec.gov/submissions/CIK{cik}.json",
        headers=EDGAR_HEADERS, timeout=30,
    )
    resp.raise_for_status()
    filings = resp.json().get("filings", {}).get("recent", {})
    return [
        {
            "accessionNumber": filings["accessionNumber"][i],
            "filingDate":      filings["filingDate"][i],
            "form":            form,
            "primaryDocument": (filings.get("primaryDocument") or [None])[i],
        }
        for i, form in enumerate(filings.get("form", []))
        if form == form_type
    ]


def fetch_filing_text(ticker: str, accession_number: str, primary_doc: Optional[str]) -> str:
    """Downloads and strips HTML from the primary document of a given accession number."""
    cik = TICKER_TO_CIK[ticker.upper()].lstrip("0")
    acc_no_dashes = accession_number.replace("-", "")
    base = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc_no_dashes}"
    if primary_doc:
        resp = requests.get(f"{base}/{primary_doc}", headers=EDGAR_HEADERS, timeout=60)
        if resp.ok:
            return _strip_html(resp.text)
    # Fallback: parse the filing index page for the largest .htm document
    index_url = f"{base}/{accession_number}-index.htm"
    resp = requests.get(index_url, headers=EDGAR_HEADERS, timeout=30)
    links = re.findall(r'href="(/Archives[^"]+\.htm)"', resp.text, re.IGNORECASE)
    if links:
        resp2 = requests.get(f"https://www.sec.gov{links[0]}", headers=EDGAR_HEADERS, timeout=60)
        return _strip_html(resp2.text)
    raise RuntimeError(f"Could not fetch filing for {ticker} {accession_number}")


def scrape_filing(ticker: str, filing_year: str, form_type: str = "10-K") -> Dict:
    """
    Finds and downloads the primary SEC filing document for a given ticker/year/form from EDGAR.
    Tries the exact calendar year first, then year+1 (common for 10-Ks filed in Jan/Feb).
    """
    filings = fetch_edgar_filings_list(ticker, form_type)
    match = next((f for f in filings if f["filingDate"].startswith(str(filing_year))), None)
    if match is None:
        match = next((f for f in filings if f["filingDate"].startswith(str(int(filing_year) + 1))), None)
    if match is None:
        raise RuntimeError(f"No {form_type} found for {ticker} FY{filing_year} on EDGAR.")
    text = fetch_filing_text(ticker, match["accessionNumber"], match.get("primaryDocument"))
    drift_log(f"  [EDGAR] Downloaded {ticker} {form_type} {filing_year} "
          f"({len(text):,} chars, filed {match['filingDate']})", verbose_only=True)
    return {
        "ticker":           ticker,
        "filing_year":      filing_year,
        "form_type":        form_type,
        "filing_date":      match["filingDate"],
        "accession_number": match["accessionNumber"],
        "text":             text,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Chunker + ChromaDB Ingestor
# ─────────────────────────────────────────────────────────────────────────────

DRIFT_CHUNK_WORDS   = 800
DRIFT_CHUNK_OVERLAP = 100


def chunk_filing(filing: Dict) -> List[Dict]:
    """Splits a raw filing text into overlapping word-window chunks matching sec_chunks.jsonl schema."""
    words   = filing["text"].split()
    company = COMPANY_NAMES_MAP.get(filing["ticker"], filing["ticker"])
    chunks, start, idx = [], 0, 0
    while start < len(words):
        end  = min(start + DRIFT_CHUNK_WORDS, len(words))
        text = " ".join(words[start:end])
        cid  = f"{filing['ticker']}_{filing['form_type']}_{filing['filing_year']}_{idx:04d}_drift"
        contextual = (
            f"Company: {company} ({filing['ticker']})\n"
            f"Filing: {filing['form_type']} | Date: {filing['filing_date']} | Year: {filing['filing_year']}\n"
            f"Section: auto-chunked\n"
            f"Content: {text}"
        )
        chunks.append({
            "chunk_id":                cid,
            "ticker":                  filing["ticker"],
            "company_name":            company,
            "form_type":               filing["form_type"],
            "filing_year":             str(filing["filing_year"]),
            "filing_date":             filing["filing_date"],
            "accession_number":        filing["accession_number"],
            "section_title":           "auto-chunked",
            "chunk_index":             idx,
            "text":                    text,
            "contextual_text":         contextual,
            "word_count":              end - start,
            "source":                  "drift_ingested",
            "flag_low_value_combined": False,
        })
        start += DRIFT_CHUNK_WORDS - DRIFT_CHUNK_OVERLAP
        idx   += 1
    drift_log(f"  [Chunker] {len(chunks)} chunks from {filing['ticker']} {filing['form_type']} {filing['filing_year']}", verbose_only=True)
    return chunks


def ingest_to_chroma(chunks: List[Dict], corpus_index: CorpusIndex) -> int:
    """
    Embeds and upserts new chunks into the CorpusIndex's existing ChromaDB collection.
    Skips chunks whose IDs are already present (idempotent).
    """
    collection = corpus_index.chroma_col
    existing   = set(collection.get(ids=[c["chunk_id"] for c in chunks])["ids"])
    new        = [c for c in chunks if c["chunk_id"] not in existing]
    if not new:
        drift_log("  [Ingestor] All chunks already present — skipping.", verbose_only=True)
        return 0
    texts = [c["contextual_text"] for c in new]
    embs  = dense_model.encode(texts, batch_size=32, show_progress_bar=False).tolist()
    metas = [{k: v for k, v in c.items() if k not in ("text", "contextual_text")} for c in new]
    collection.upsert(
        ids=       [c["chunk_id"] for c in new],
        embeddings=embs,
        documents= texts,
        metadatas= metas,
    )
    drift_log(f"  [Ingestor] Added {len(new)} chunks to ChromaDB.", verbose_only=True)
    return len(new)


def update_corpus_index(corpus_index: CorpusIndex, new_chunks: List[Dict]) -> None:
    """
    Hot-patches the live CorpusIndex with freshly scraped chunks.
    Appends rows to the df, rebuilds BM25, and updates the str→row lookup —
    all in-memory so the current session benefits without a kernel restart.
    Also appends new chunks to sec_chunks.jsonl so future sessions load them automatically.
    """
    base_len = len(corpus_index.df)
    new_rows = []
    for i, c in enumerate(new_chunks):
        doc_name   = f"{c['ticker']}_{c['form_type']}_{c['filing_date'][:10]}"
        contextual = c["contextual_text"]
        new_rows.append({
            "chunk_id":         base_len + i,
            "chunk_id_str":     c["chunk_id"],
            "ticker":           c["ticker"],
            "company":          c["company_name"],
            "doc_name":         doc_name,
            "filing_year":      int(c["filing_year"]),
            "form_type":        c["form_type"],
            "page_num":         c["chunk_index"],
            "raw_chunk":        c["text"],
            "contextual_chunk": contextual,
            "bm25_tokens":      contextual.lower().split(),
        })
    new_df = pd.DataFrame(new_rows)
    corpus_index.df   = pd.concat([corpus_index.df, new_df], ignore_index=True)
    corpus_index.bm25 = BM25Okapi(corpus_index.df["bm25_tokens"].tolist())
    # Update the str→row lookup used by dense_search for BM25↔ChromaDB cross-referencing
    for i, c in enumerate(new_chunks):
        corpus_index._str_to_row[c["chunk_id"]] = base_len + i
    # Persist to JSONL so future sessions load new data without re-scraping
    with open(CONFIG["sec_chunks_path"], "a", encoding="utf-8") as f:
        for c in new_chunks:
            f.write(json.dumps(c) + "\n")
    drift_log(f"  [BM25] Index rebuilt: {len(corpus_index.df):,} total chunks.", verbose_only=True)


In [14]:
# ---------------------------------------------------------------------------
# LangGraph state + nodes
# ---------------------------------------------------------------------------

class GraphState(TypedDict, total=False):
    question:              str
    index:                 CorpusIndex
    # Planner outputs
    rewritten_query:       str
    sub_queries:           List[Dict]   # list of SubQuery dicts
    needs_decomposition:   bool
    # Retrieval
    retrieved_chunks:      List[RetrievedChunk]
    retrieved_doc_names:   List[str]
    retrieval_sanity_failed: bool
    # Context eval
    context_retries:       int
    context_eval_relevant: bool
    # Generation
    generated_answer:      str
    # Critic
    critic_decision:       str
    critic_reason:         str
    # Repair
    repair_used:           bool
    repair_decision:       str
    repair_reason:         str
    needs_new_retrieval:   bool
    repair_retrieval_count: int
    is_repair_retrieval:   bool
    # Final
    final_answer:          str
    # Drift detection
    drift_pending_scrapes: List[Tuple]  # [(ticker, filing_year, form_type), ...]
    drift_triggered:       bool
    ingestion_just_ran:    bool
    use_seeded_retrieval: bool


# ── Proposal 1: Planner node ─────────────────────────────────────────────────

def node_query_planner(state: GraphState) -> GraphState:
    if state.get("use_seeded_retrieval", False):
        existing_query = state.get("rewritten_query", state["question"])
        sub_queries = state.get("sub_queries") or [{"query": existing_query}]
        return {
            "rewritten_query": existing_query,
            "sub_queries": sub_queries,
            "needs_decomposition": state.get("needs_decomposition", False),
        }

    result = safe_invoke_structured(
        "agent",
        PlannerOutput,
        planner_prompt,
        {"question": state["question"]},
        PlannerOutput(
            needs_decomposition=False,
            sub_queries=[SubQuery(query=state["question"])],
        ),
        task_name="planner",
    )
    sub_queries = []
    for sq in result.sub_queries:
        sq_dict = sq.model_dump()
        query = (sq_dict.get("query") or "").strip()
        if not query:
            parts = [state["question"]]
            if sq_dict.get("ticker"):
                parts.append(f"ticker {sq_dict['ticker']}")
            if sq_dict.get("filing_year"):
                parts.append(f"filing year {sq_dict['filing_year']}")
            if sq_dict.get("form_type"):
                parts.append(f"form {sq_dict['form_type']}")
            query = " | ".join(parts)
        sq_dict["query"] = cleanup_planner_query(
            query,
            ticker=sq_dict.get("ticker"),
            filing_year=sq_dict.get("filing_year"),
            form_type=sq_dict.get("form_type"),
        )
        sub_queries.append(sq_dict)
    rewritten_query = sub_queries[0]["query"] if sub_queries else state["question"]
    n = len(sub_queries)
    label = "decomposed" if result.needs_decomposition else "single"
    print(f"  [Planner] {label} | {n} sub-quer{'ies' if n > 1 else 'y'}")
    if result.needs_decomposition:
        for sq in sub_queries:
            print(f"    → {sq['query']}  (ticker={sq.get('ticker')}, year={sq.get('filing_year')})")
    return {
        "rewritten_query":     rewritten_query,
        "sub_queries":         sub_queries,
        "needs_decomposition": result.needs_decomposition,
    }


# ── Updated hybrid retriever (multi-query aware) ─────────────────────────────

def node_hybrid_retriever(state: GraphState) -> GraphState:
    if state.get("use_seeded_retrieval", False) and state.get("retrieved_chunks"):
        retrieved = state.get("retrieved_chunks", [])
        sub_queries = state.get("sub_queries", [])
        sanity_failed = fails_retrieval_sanity_check(state["question"], sub_queries, retrieved)
        return {
            "rewritten_query": state.get("rewritten_query", state["question"]),
            "retrieved_chunks": retrieved,
            "retrieved_doc_names": state.get("retrieved_doc_names", extract_retrieved_doc_names(retrieved)),
            "retrieval_sanity_failed": sanity_failed,
        }

    sub_queries = state.get("sub_queries", [])

    if len(sub_queries) <= 1:
        query = state.get("rewritten_query", state["question"])
        sq = sub_queries[0] if sub_queries else {}
        retrieval = run_advanced_retrieval(
            state["index"],
            query,
            meta_override={
                "ticker": sq.get("ticker"),
                "filing_year": sq.get("filing_year"),
                "form_type": sq.get("form_type"),
            },
            rerank_top_k=CONFIG["rerank_top_k"],
        )
        retrieved = retrieval["retrieved_chunks"]
        rewritten_query = retrieval["rewritten"]
    else:
        per_k = CONFIG["decomposition_top_k_per_subquery"]
        merged: Dict[str, RetrievedChunk] = {}
        for sq in sub_queries:
            retrieval = run_advanced_retrieval(
                state["index"],
                sq["query"],
                meta_override={
                    "ticker": sq.get("ticker"),
                    "filing_year": sq.get("filing_year"),
                    "form_type": sq.get("form_type"),
                },
                rerank_top_k=per_k,
            )
            for chunk in retrieval["retrieved_chunks"]:
                key = chunk.chunk_id
                if key not in merged or chunk.score > merged[key].score:
                    merged[key] = chunk
        retrieved = sorted(merged.values(), key=lambda chunk: chunk.score, reverse=True)[:CONFIG["rerank_top_k"]]
        rewritten_query = state.get("rewritten_query", state["question"])

    sanity_failed = fails_retrieval_sanity_check(state["question"], sub_queries, retrieved)
    return {
        "rewritten_query":         rewritten_query,
        "retrieved_chunks":        retrieved,
        "retrieved_doc_names":     extract_retrieved_doc_names(retrieved),
        "retrieval_sanity_failed": sanity_failed,
    }


def node_context_evaluator(state: GraphState) -> GraphState:
    if state.get("retrieval_sanity_failed", False):
        return {"context_eval_relevant": False, "context_retries": state.get("context_retries", 0)}
    max_chunks, max_chars = get_model_aware_context_limits(
        "agent", "context_eval", CONFIG["control_max_context_chunks"], CONFIG["control_max_context_chars"]
    )
    context = format_retrieved_context(
        state.get("retrieved_chunks", []),
        max_chunks=max_chunks,
        max_chars=max_chars,
    )
    result = safe_invoke_structured(
        "agent",
        ContextEvalOutput,
        context_eval_prompt,
        {"question": state["question"], "context": context},
        ContextEvalOutput(relevant=True, reason="Fallback accept"),
        task_name="context_eval",
    )
    return {"context_eval_relevant": result.relevant, "context_retries": state.get("context_retries", 0)}


def route_context(state: GraphState) -> str:
    if state.get("context_eval_relevant", True):
        return "generator"
    if state.get("context_retries", 0) < CONFIG["max_context_retries"]:
        return "retry_retrieval"
    return "generator"


def node_increment_context_retry(state: GraphState) -> GraphState:
    return {"context_retries": state.get("context_retries", 0) + 1}


def node_generator(state: GraphState) -> GraphState:
    max_chunks, max_chars = get_model_aware_context_limits(
        "generator", "generator", CONFIG["generator_max_context_chunks"], CONFIG["generator_max_context_chars"]
    )
    few_shot_examples = select_few_shot_examples(state["question"])
    context = format_retrieved_context(
        state.get("retrieved_chunks", []),
        max_chunks=max_chunks,
        max_chars=max_chars,
    )
    raw = safe_invoke_llm(
        "generator",
        generator_prompt,
        {"question": state["question"], "context": context, "few_shot_examples": few_shot_examples},
        task_name="generator",
    )
    answer = normalize_answer_text(raw)
    return {"generated_answer": answer, "final_answer": answer}


# ── Proposal 2: Updated critic node + routing ─────────────────────────────────

def node_critic(state: GraphState) -> GraphState:
    max_chunks, max_chars = get_model_aware_context_limits(
        "agent", "critic", CONFIG["control_max_context_chunks"], CONFIG["control_max_context_chars"]
    )
    context = format_retrieved_context(
        state.get("retrieved_chunks", []),
        max_chunks=max_chunks,
        max_chars=max_chars,
    )
    result = safe_invoke_structured(
        "agent",
        CriticOutput,
        critic_prompt,
        {
            "question": state["question"],
            "context":  context,
            "answer":   state.get("generated_answer", ""),
        },
        CriticOutput(decision="accept", reason="Fallback accept"),
        task_name="critic",
    )
    updates: GraphState = {"critic_decision": result.decision, "critic_reason": result.reason}
    # Proposal 2: if data is absent, set the final answer immediately so we exit cleanly
    if result.decision == "insufficient_data":
        updates["final_answer"] = (
            f"Insufficient data: The required information could not be found in the "
            f"retrieved SEC filings. ({result.reason})"
        )
    return updates


def route_critic(state: GraphState) -> str:
    if state.get("is_repair_retrieval", False):
        return "end"
    decision = state.get("critic_decision", "accept")
    if decision == "repair":
        return "repair"
    if decision == "insufficient_data":
        return "insufficient_data"
    return "end"


def node_repair(state: GraphState) -> GraphState:
    max_chunks, max_chars = get_model_aware_context_limits(
        "agent", "repair", CONFIG["control_max_context_chunks"], CONFIG["control_max_context_chars"]
    )
    context = format_retrieved_context(
        state.get("retrieved_chunks", []),
        max_chunks=max_chunks,
        max_chars=max_chars,
    )
    result = safe_invoke_structured(
        "agent",
        RepairOutput,
        repair_prompt,
        {
            "question": state["question"],
            "context":  context,
            "answer":   state.get("generated_answer", ""),
            "critique": state.get("critic_reason", "Needs repair"),
        },
        RepairOutput(
            decision="accept",
            revised_answer=state.get("generated_answer", ""),
            needs_new_retrieval=False,
            reason="Fallback repair",
        ),
        task_name="repair",
    )
    count = state.get("repair_retrieval_count", 0)
    if result.needs_new_retrieval:
        count += 1
    return {
        "repair_used":           True,
        "repair_decision":       result.decision,
        "repair_reason":         result.reason,
        "needs_new_retrieval":   result.needs_new_retrieval,
        "repair_retrieval_count": count,
        "final_answer":          normalize_answer_text(result.revised_answer),
    }


def route_repair(state: GraphState) -> str:
    if state.get("needs_new_retrieval", False) and state.get("repair_retrieval_count", 0) <= CONFIG["max_repair_retrievals"]:
        return "re_retrieve"
    return "end"


def node_mark_repair_retrieval(state: GraphState) -> GraphState:
    return {"is_repair_retrieval": True}

# ── Drift detection nodes (Proposal 3) ───────────────────────────────────────

def node_drift_detector(state: GraphState) -> GraphState:
    """
    Called after the critic returns 'insufficient_data'.
    Extracts the ticker/year/form from the planner's sub-queries, logs the miss
    to DriftTracker, then checks whether the scrape threshold has been crossed.
    """
    sub_queries = state.get("sub_queries", [])
    ticker, filing_year, form_type = None, None, None
    if sub_queries:
        sq         = sub_queries[0]
        ticker     = sq.get("ticker")
        filing_year = str(sq.get("filing_year")) if sq.get("filing_year") else None
        form_type  = sq.get("form_type") or "10-K"

    drift_tracker.log_miss(
        question    = state["question"],
        ticker      = ticker,
        filing_year = filing_year,
        form_type   = form_type,
    )

    pending = drift_tracker.get_pending_scrapes()
    if pending:
        drift_log(f"  [DriftDetector] Pending scrapes: {pending}")

    return {
        "drift_pending_scrapes": pending,
        "drift_triggered":       len(pending) > 0,
        "ingestion_just_ran":    False,
    }


def route_drift(state: GraphState) -> str:
    """Scrape if threshold crossed and data is available on EDGAR; otherwise exit cleanly."""
    if state.get("drift_triggered") and state.get("drift_pending_scrapes"):
        return "scrape_and_ingest"
    return "end"


def node_scrape_and_ingest(state: GraphState) -> GraphState:
    """
    Scrapes each pending (ticker, year, form_type) from EDGAR, chunks the text,
    upserts into ChromaDB, and hot-patches the live CorpusIndex's BM25 index.
    Clears critic state so the graph retries retrieval with the fresh data.
    """
    corpus_index = state["index"]
    ingested_any = False

    for ticker, filing_year, form_type in state.get("drift_pending_scrapes", []):
        try:
            filing = scrape_filing(ticker, str(filing_year), form_type)
            chunks = chunk_filing(filing)
            added  = ingest_to_chroma(chunks, corpus_index)
            if added > 0:
                update_corpus_index(corpus_index, chunks)
                ingested_any = True
            drift_tracker.mark_ingested(ticker, str(filing_year), form_type)
        except Exception as e:
            drift_tracker.mark_failed(ticker, str(filing_year), form_type, str(e))
            if CONFIG.get("verbose_drift_failures", True):
                drift_log(f"  [ScrapeIngest] Failed for {ticker} {filing_year}: {e}")

    return {
        "ingestion_just_ran":  ingested_any,
        "context_retries":     0,
        "critic_decision":     None,
        "is_repair_retrieval": False,
    }


def route_after_scrape(state: GraphState) -> str:
    """If new data was ingested, route back to retrieval for a second attempt; otherwise exit."""
    if state.get("ingestion_just_ran"):
        return "retry_retrieval"
    return "end"


In [15]:
# ---------------------------------------------------------------------------
# Graph construction
# Entry point changed: query_planner replaces query_rewriter
# ---------------------------------------------------------------------------

def build_agentic_graph():
    g = StateGraph(GraphState)

    g.add_node("query_planner",          node_query_planner)
    g.add_node("hybrid_retriever",        node_hybrid_retriever)
    g.add_node("context_evaluator",       node_context_evaluator)
    g.add_node("increment_context_retry", node_increment_context_retry)
    g.add_node("generator",               node_generator)
    g.add_node("critic",                  node_critic)
    g.add_node("repair",                  node_repair)
    g.add_node("mark_repair_retrieval",   node_mark_repair_retrieval)
    g.add_node("drift_detector",          node_drift_detector)
    g.add_node("scrape_and_ingest",       node_scrape_and_ingest)

    def route_after_planner(state: GraphState) -> str:
        if state.get("use_seeded_retrieval", False) and state.get("retrieved_chunks"):
            return "context_evaluator"
        return "hybrid_retriever"

    g.set_entry_point("query_planner")
    g.add_conditional_edges(
        "query_planner",
        route_after_planner,
        {"hybrid_retriever": "hybrid_retriever", "context_evaluator": "context_evaluator"},
    )
    g.add_edge("hybrid_retriever",        "context_evaluator")
    g.add_conditional_edges(
        "context_evaluator",
        route_context,
        {"generator": "generator", "retry_retrieval": "increment_context_retry"},
    )
    g.add_edge("increment_context_retry", "hybrid_retriever")
    g.add_edge("generator",               "critic")
    g.add_conditional_edges(
        "critic",
        route_critic,
        {"repair": "repair", "end": END, "insufficient_data": "drift_detector"},
    )
    g.add_conditional_edges(
        "drift_detector",
        route_drift,
        {"scrape_and_ingest": "scrape_and_ingest", "end": END},
    )
    g.add_conditional_edges(
        "scrape_and_ingest",
        route_after_scrape,
        {"retry_retrieval": "hybrid_retriever", "end": END},
    )
    g.add_conditional_edges(
        "repair",
        route_repair,
        {"re_retrieve": "mark_repair_retrieval", "end": END},
    )
    g.add_edge("mark_repair_retrieval",   "hybrid_retriever")

    return g.compile()


agentic_graph = build_agentic_graph()
print("Agentic graph compiled (with drift detection).")

Agentic graph compiled (with drift detection).


In [16]:
# ---------------------------------------------------------------------------
# Metrics helpers
# ---------------------------------------------------------------------------

def normalize_answer_text(answer: Any) -> str:
    if answer is None:
        return ""
    if isinstance(answer, str):
        return answer.strip()
    if isinstance(answer, list):
        return " ".join(str(x) for x in answer if x is not None).strip()
    if isinstance(answer, dict):
        for key in ["final_answer", "answer", "text", "content", "generated_answer"]:
            if key in answer:
                return normalize_answer_text(answer[key])
        return str(answer).strip()
    return str(answer).strip()


def normalize_for_eval(text: Any) -> str:
    text = normalize_answer_text(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def token_f1_score(prediction: Any, reference: Any) -> float:
    pred_tokens = normalize_for_eval(prediction).split()
    ref_tokens  = normalize_for_eval(reference).split()
    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(ref_tokens)
    n_same = sum(common.values())
    if n_same == 0:
        return 0.0
    p = n_same / len(pred_tokens)
    r = n_same / len(ref_tokens)
    return 2 * p * r / (p + r)


def token_precision_recall(prediction: Any, reference: Any) -> Tuple[float, float]:
    pred_tokens = normalize_for_eval(prediction).split()
    ref_tokens  = normalize_for_eval(reference).split()
    if not pred_tokens and not ref_tokens:
        return 1.0, 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0, 0.0
    common = Counter(pred_tokens) & Counter(ref_tokens)
    n_same = sum(common.values())
    if n_same == 0:
        return 0.0, 0.0
    return n_same / len(pred_tokens), n_same / len(ref_tokens)


def _extract_numbers(text: str) -> List[float]:
    text = text.lower()
    for word, mult in [("trillion", 1e12), ("billion", 1e9), ("million", 1e6), ("thousand", 1e3)]:
        text = re.sub(
            rf"([\d,.]+)\s*{word}",
            lambda m, mult=mult: str(float(m.group(1).replace(",", "")) * mult),
            text,
        )
    numbers = []
    for tok in re.findall(r"\(?\$?[\d,]+\.?\d*\)?", text):
        negative = tok.startswith("(") and tok.endswith(")")
        try:
            val = float(re.sub(r"[^\d.]", "", tok))
            numbers.append(-val if negative else val)
        except ValueError:
            pass
    return numbers


def numerical_accuracy_score(prediction: Any, reference: Any, tolerance: float = 0.01) -> float:
    pred_nums = _extract_numbers(normalize_answer_text(prediction))
    ref_nums  = _extract_numbers(normalize_answer_text(reference))
    if not ref_nums:
        return 1.0
    if not pred_nums:
        return 0.0
    for ref_val in ref_nums:
        for pred_val in pred_nums:
            if ref_val == 0 and pred_val == 0:
                return 1.0
            if ref_val != 0 and abs(pred_val - ref_val) / abs(ref_val) <= tolerance:
                return 1.0
    return 0.0


def compute_decision_from_text(answer: Any) -> str:
    lower = normalize_answer_text(answer).lower()
    if any(x in lower for x in ["insufficient data", "cannot answer", "not available", "not found", "do not know"]):
        return "warn"
    if any(x in lower for x in ["refuse", "cannot comply"]):
        return "refuse"
    return "answer"


def retrieval_metrics(
    retrieved_chunks: Any,
    evidence_chunk_id: str = "",
    ticker: str = "",
    filing_year: Any = None,
    form_type: str = "",
) -> Tuple[float, float, float]:
    chunks = retrieved_chunks if isinstance(retrieved_chunks, list) else []
    if not chunks:
        return 0.0, 0.0, 0.0
    retrieved_ids = [getattr(chunk, 'chunk_id', None) for chunk in chunks]
    recall = 1.0 if evidence_chunk_id and evidence_chunk_id in retrieved_ids else 0.0
    precision = (1.0 / len(chunks)) if recall > 0 else 0.0
    relevant_count = 0
    target_ticker = str(ticker or '').strip().upper()
    target_form = str(form_type or '').strip().upper()
    target_year = int(filing_year) if filing_year not in (None, '', float('nan')) and pd.notna(filing_year) else None
    for chunk in chunks:
        same_ticker = not target_ticker or str(getattr(chunk, 'ticker', '')).strip().upper() == target_ticker
        same_form = not target_form or str(getattr(chunk, 'form_type', '')).strip().upper() == target_form
        same_year = target_year is None or getattr(chunk, 'filing_year', None) == target_year
        if same_ticker and same_form and same_year:
            relevant_count += 1
    relevancy = relevant_count / len(chunks) if chunks else 0.0
    return precision, recall, relevancy


def correct_answer_binary(
    expected_decision: str,
    llm_correctness_score: Any,
    numerical_accuracy: Any,
    token_f1: Any,
    task_completion: float,
) -> int:
    expected_decision = str(expected_decision or '').strip().lower()
    if expected_decision != 'answer':
        return int(task_completion >= 1.0)
    if llm_correctness_score is not None and pd.notna(llm_correctness_score):
        return int(float(llm_correctness_score) >= 0.8)
    if numerical_accuracy is not None and pd.notna(numerical_accuracy) and float(numerical_accuracy) >= 1.0:
        return 1
    if token_f1 is not None and pd.notna(token_f1) and float(token_f1) >= 0.8:
        return 1
    return 0


def build_agent_route_trace(state: Dict[str, Any]) -> str:
    steps = ["planner"]
    if state.get("needs_decomposition"):
        steps.append("decompose")
    steps.append("retrieve")
    if state.get("retrieval_sanity_failed"):
        steps.append("retrieval_sanity_failed")
    if (state.get("context_retries") or 0) > 0:
        steps.append(f"context_retry={state.get('context_retries', 0)}")
    steps.append("generate")
    critic_decision = state.get("critic_decision")
    if critic_decision:
        steps.append(f"critic={critic_decision}")
    if state.get("repair_used"):
        steps.append(f"repair={state.get('repair_decision') or 'used'}")
    if (state.get("repair_retrieval_count") or 0) > 0:
        steps.append(f"repair_retrievals={state.get('repair_retrieval_count', 0)}")
    if state.get("drift_triggered"):
        steps.append("drift_triggered")
    pending = state.get("drift_pending_scrapes") or []
    if pending:
        steps.append(f"pending_scrapes={len(pending)}")
    if state.get("ingestion_just_ran"):
        steps.append("ingested_new_filing")
    return " -> ".join(steps)


In [17]:
# ---------------------------------------------------------------------------
# Pipeline runners — all now accept the global index, no per-question corpus
# ---------------------------------------------------------------------------

def _normalize_prior_question_id(value: Any) -> str:
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return ""
    try:
        return f"SEC_{int(float(text)):03d}"
    except Exception:
        return text


def load_prior_advanced_results(eval_df: pd.DataFrame) -> Dict[str, Any]:
    if not CONFIG.get("reuse_prior_advanced_results", True):
        print("Prior advanced-results reuse disabled; LangGraph will run advanced_rag live.")
        return {}

    candidate_paths = []
    configured_path = str(CONFIG.get("prior_advanced_results_path", "")).strip()
    if configured_path:
        candidate_paths.append(Path(configured_path))
    candidate_paths.append(Path(CONFIG["results_dir"]) / "advanced_results.csv")
    candidate_paths.append(PROJECT_ROOT / "baseline_advanced_rag" / "results" / "advanced_results.csv")

    existing_paths = []
    seen = set()
    for candidate in candidate_paths:
        try:
            resolved = candidate.expanduser().resolve()
        except Exception:
            resolved = candidate.expanduser()
        key = str(resolved).lower()
        if key in seen:
            continue
        seen.add(key)
        if resolved.exists():
            existing_paths.append(resolved)

    if not existing_paths:
        message = (
            "No prior advanced_results.csv found. LangGraph will fall back to running advanced_rag live."
        )
        if CONFIG.get("require_prior_advanced_results", False):
            raise FileNotFoundError(message)
        print(message)
        return {}

    selected_path = existing_paths[0]
    prior_df = pd.read_csv(selected_path).copy()
    if prior_df.empty:
        message = f"Prior advanced results at {selected_path} are empty; falling back to live advanced_rag."
        if CONFIG.get("require_prior_advanced_results", False):
            raise ValueError(message)
        print(message)
        return {}

    if "predicted_answer" not in prior_df.columns and "final_answer" in prior_df.columns:
        prior_df["predicted_answer"] = prior_df["final_answer"]
    prior_df["normalized_id"] = prior_df.get("question_id", pd.Series(index=prior_df.index, dtype=object)).apply(_normalize_prior_question_id)
    prior_df["question_normalized"] = prior_df.get("question", pd.Series(index=prior_df.index, dtype=object)).fillna("").astype(str).str.strip()

    for col in ["correctness", "faithfulness", "claims_covered", "n_chunks"]:
        if col in prior_df.columns:
            prior_df[col] = pd.to_numeric(prior_df[col], errors="coerce")

    by_id = {}
    for _, prior_row in prior_df.iterrows():
        normalized_id = prior_row.get("normalized_id", "")
        if normalized_id and normalized_id not in by_id:
            by_id[normalized_id] = prior_row.copy()

    by_question = {}
    for _, prior_row in prior_df.iterrows():
        question_key = prior_row.get("question_normalized", "")
        if question_key and question_key not in by_question:
            by_question[question_key] = prior_row.copy()

    eval_ids = eval_df["id"].fillna("").astype(str).str.strip()
    eval_questions = eval_df["question"].fillna("").astype(str).str.strip()
    matched_by_id = int(eval_ids.isin(set(by_id.keys())).sum())
    matched_by_question = int(eval_questions.isin(set(by_question.keys())).sum())

    print(
        f"Loaded prior advanced results: {selected_path} | rows={len(prior_df)} | "
        f"matches_by_id={matched_by_id}/{len(eval_df)} | matches_by_question={matched_by_question}/{len(eval_df)}"
    )

    return {
        "path": str(selected_path),
        "df": prior_df,
        "by_id": by_id,
        "by_question": by_question,
    }


def get_prior_advanced_row(eval_row: pd.Series, prior_cache: Dict[str, Any]) -> Optional[pd.Series]:
    if not prior_cache:
        return None

    qid = str(eval_row.get("id", "")).strip()
    if qid and qid in prior_cache.get("by_id", {}):
        return prior_cache["by_id"][qid]

    question = str(eval_row.get("question", "")).strip()
    if question and question in prior_cache.get("by_question", {}):
        return prior_cache["by_question"][question]

    return None


def build_advanced_output_from_prior(eval_row: pd.Series, prior_row: pd.Series) -> Dict[str, Any]:
    answer = normalize_answer_text(
        prior_row.get("predicted_answer", prior_row.get("final_answer", ""))
    )
    rewritten_query = normalize_answer_text(
        prior_row.get("rewritten_query", eval_row.get("question", ""))
    )

    correctness = prior_row.get("correctness")
    faithfulness = prior_row.get("faithfulness")
    claims_covered = prior_row.get("claims_covered")
    n_chunks = prior_row.get("n_chunks")

    return {
        "pipeline": "advanced_rag",
        "rewritten_query": rewritten_query or str(eval_row.get("question", "")),
        "retrieved_chunks": [],
        "retrieved_doc_names": [],
        "context_retries": 0,
        "context_eval_relevant": None,
        "generated_answer": answer,
        "critic_decision": None,
        "repair_used": False,
        "repair_decision": None,
        "final_answer": answer,
        "latency_sec": None if pd.isna(prior_row.get("latency_sec")) else float(prior_row.get("latency_sec")),
        "needs_decomposition": False,
        "repair_retrieval_count": 0,
        "llm_models_used": {},
        "generator_llm": "precomputed:advanced_rag",
        "agent_llm": "precomputed:advanced_rag",
        "judge_llm": "precomputed:advanced_rag",
        "llm_correctness_score": None if pd.isna(correctness) else float(correctness),
        "llm_claims_covered": None if pd.isna(claims_covered) else float(claims_covered),
        "llm_correctness_reason": str(prior_row.get("corr_explanation", "Loaded from prior advanced results")).strip(),
        "llm_faithfulness_score": None if pd.isna(faithfulness) else float(faithfulness),
        "llm_faithfulness_reason": str(prior_row.get("faith_explanation", "Loaded from prior advanced results")).strip(),
        "output_source": "prior_advanced_results",
        "prior_advanced_results_path": str(prior_cache_path) if (prior_cache_path := prior_row.get("_source_path")) else None,
        "precomputed_n_chunks": None if pd.isna(n_chunks) else int(n_chunks),
    }


def get_advanced_output(eval_row: pd.Series, index: CorpusIndex, prior_cache: Dict[str, Any]) -> Dict[str, Any]:
    prior_row = get_prior_advanced_row(eval_row, prior_cache)
    if prior_row is not None:
        prior_copy = prior_row.copy()
        prior_copy["_source_path"] = prior_cache.get("path")
        return build_advanced_output_from_prior(eval_row, prior_copy)
    return run_advanced_rag(str(eval_row["question"]), index)


def run_simple_rag(question: str, index: CorpusIndex) -> Dict[str, Any]:
    start = time.time()
    retrieved = index.dense_search(question, top_k=CONFIG["rerank_top_k"])
    few_shot_examples = select_few_shot_examples(question)
    gen_chunks, gen_chars = get_model_aware_context_limits(
        "generator", "generator", CONFIG["generator_max_context_chunks"], CONFIG["generator_max_context_chars"]
    )
    context   = format_retrieved_context(
        retrieved,
        max_chunks=gen_chunks,
        max_chars=gen_chars,
    )
    raw       = safe_invoke_llm(
        "generator", generator_prompt, {"question": question, "context": context, "few_shot_examples": few_shot_examples}, task_name="generator"
    )
    answer = normalize_answer_text(raw)
    llm_models_used = get_task_model_snapshot(["generator"])
    return {
        "pipeline": "simple_rag",
        "rewritten_query": question,
        "retrieved_chunks": retrieved,
        "retrieved_doc_names": extract_retrieved_doc_names(retrieved),
        "context_retries": 0, "context_eval_relevant": None,
        "generated_answer": answer,
        "critic_decision": None, "repair_used": False, "repair_decision": None,
        "final_answer": answer, "latency_sec": time.time() - start,
        "needs_decomposition": False, "repair_retrieval_count": 0,
        "llm_models_used": llm_models_used,
        "generator_llm": llm_models_used.get("generator"),
        "agent_llm": None,
        "judge_llm": None,
    }


def run_advanced_rag(question: str, index: CorpusIndex) -> Dict[str, Any]:
    start = time.time()
    retrieval = run_advanced_retrieval(index, question)
    retrieved = retrieval["retrieved_chunks"]
    answer = normalize_answer_text(generate_with_citations(question, retrieved))
    llm_models_used = get_task_model_snapshot(["rewrite", "multi_query", "generator"])
    return {
        "pipeline": "advanced_rag",
        "rewritten_query": retrieval["rewritten"],
        "retrieved_chunks": retrieved,
        "retrieved_doc_names": extract_retrieved_doc_names(retrieved),
        "context_retries": 0,
        "context_eval_relevant": None,
        "generated_answer": answer,
        "critic_decision": None,
        "repair_used": False,
        "repair_decision": None,
        "final_answer": answer,
        "latency_sec": time.time() - start,
        "needs_decomposition": False,
        "repair_retrieval_count": 0,
        "llm_models_used": llm_models_used,
        "generator_llm": llm_models_used.get("generator"),
        "agent_llm": llm_models_used.get("rewrite") or llm_models_used.get("multi_query"),
        "judge_llm": None,
        "output_source": "live_advanced_rag",
    }


def run_agentic_rag(question: str, index: CorpusIndex, advanced_output: Dict[str, Any] = None) -> Dict[str, Any]:
    start = time.time()
    advanced_output = advanced_output or run_advanced_rag(question, index)
    seeded_chunks = advanced_output.get("retrieved_chunks") or []
    seeded_doc_names = advanced_output.get("retrieved_doc_names") or extract_retrieved_doc_names(seeded_chunks)
    seeded_retrieval_available = bool(seeded_chunks)
    state: GraphState = {
        "question":               question,
        "index":                  index,
        "rewritten_query":        advanced_output.get("rewritten_query", question),
        "retrieved_chunks":       seeded_chunks,
        "retrieved_doc_names":    seeded_doc_names,
        "context_retries":        0,
        "repair_used":            False,
        "repair_retrieval_count": 0,
        "is_repair_retrieval":    False,
        "retrieval_sanity_failed": False,
        "sub_queries":            [{"query": advanced_output.get("rewritten_query", question)}],
        "needs_decomposition":    False,
        "drift_pending_scrapes":  [],
        "drift_triggered":        False,
        "ingestion_just_ran":     False,
        "use_seeded_retrieval":   seeded_retrieval_available,
    }
    out = agentic_graph.invoke(state)
    latency = time.time() - start

    generated = normalize_answer_text(out.get("generated_answer", ""))
    final = normalize_answer_text(out.get("final_answer", generated))
    llm_models_used = get_task_model_snapshot(["planner", "rewrite", "multi_query", "context_eval", "generator", "critic", "repair", "judge"])
    agent_result = {
        "pipeline":                "agentic_rag",
        "rewritten_query":         normalize_answer_text(out.get("rewritten_query", question)),
        "retrieved_chunks":        out.get("retrieved_chunks", []),
        "retrieved_doc_names":     out.get("retrieved_doc_names", []),
        "context_retries":         out.get("context_retries", 0),
        "context_eval_relevant":   out.get("context_eval_relevant"),
        "generated_answer":        generated,
        "critic_decision":         out.get("critic_decision"),
        "critic_reason":           out.get("critic_reason"),
        "repair_used":             out.get("repair_used", False),
        "repair_decision":         out.get("repair_decision"),
        "repair_reason":           out.get("repair_reason"),
        "repair_retrieval_count":  out.get("repair_retrieval_count", 0),
        "needs_decomposition":     out.get("needs_decomposition", False),
        "retrieval_sanity_failed": out.get("retrieval_sanity_failed", False),
        "drift_triggered":         out.get("drift_triggered", False),
        "drift_pending_scrapes":   out.get("drift_pending_scrapes", []),
        "ingestion_just_ran":      out.get("ingestion_just_ran", False),
        "final_answer":            final,
        "latency_sec":             latency,
        "llm_models_used":         llm_models_used,
        "generator_llm":           llm_models_used.get("generator"),
        "agent_llm":               llm_models_used.get("planner") or llm_models_used.get("critic") or llm_models_used.get("context_eval"),
        "judge_llm":               llm_models_used.get("judge"),
        "output_source":           "seeded_prior_advanced" if advanced_output.get("output_source") == "prior_advanced_results" else "live_agentic_rag",
        "seeded_from_prior_advanced": advanced_output.get("output_source") == "prior_advanced_results",
    }
    agent_result["agent_trace"] = build_agent_route_trace(agent_result)
    return agent_result


In [18]:
# ---------------------------------------------------------------------------
# LLM judge + results export helpers
# ---------------------------------------------------------------------------

print("LLM judge is applied during the main eval loop for every evaluated question.")

def llm_judge_correctness(question: str, reference_answer: str, candidate_answer: str) -> Tuple[float, float, str]:
    result = safe_invoke_structured(
        "judge", JudgeOutput, judge_correctness_prompt,
        {"question": question, "reference_answer": reference_answer, "candidate_answer": candidate_answer},
        JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
        task_name="judge",
    )
    return max(0.0, min(1.0, float(result.score))), max(0.0, min(1.0, float(result.claims_covered))), result.reason


def llm_judge_faithfulness(question: str, context: str, candidate_answer: str) -> Tuple[float, float, str]:
    result = safe_invoke_structured(
        "judge", JudgeOutput, judge_faithfulness_prompt,
        {"question": question, "context": context, "candidate_answer": candidate_answer},
        JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
        task_name="judge",
    )
    return max(0.0, min(1.0, float(result.score))), max(0.0, min(1.0, float(result.claims_covered))), result.reason

def build_results_input_block(
    results_df: pd.DataFrame,
    pipeline_name: str = "agentic_rag",
) -> pd.DataFrame:
    agentic_df = results_df[results_df["pipeline"] == pipeline_name].copy()
    if agentic_df.empty:
        raise ValueError(f"No rows found for pipeline={pipeline_name!r}")

    export_rows = []
    for _, row in agentic_df.sort_values("question_number").iterrows():
        retrieval_precision, retrieval_recall, retrieval_relevancy = retrieval_metrics(
            row.get("retrieved_chunks", []),
            evidence_chunk_id=row.get("evidence_chunk_id", ""),
            ticker=row.get("ticker", ""),
            filing_year=row.get("filing_year"),
            form_type=row.get("form_type", ""),
        )
        token_precision, token_recall = token_precision_recall(
            row.get("final_answer", ""), row.get("reference_answer", "")
        )

        context_str = format_retrieved_context(
            row.get("retrieved_chunks", []),
            max_chunks=CONFIG["judge_max_context_chunks"],
            max_chars=CONFIG["judge_max_context_chars"],
        ) if row.get("retrieved_chunks") else row.get("final_answer", "")

        faithfulness_score = row.get("llm_faithfulness_score")
        if faithfulness_score is None or pd.isna(faithfulness_score):
            faithfulness_score, _, _ = llm_judge_faithfulness(
                row["question"], context_str, row.get("final_answer", "")
            )

        if row.get("expected_decision", "answer") == "answer" and row.get("reference_answer", ""):
            answer_relevancy, claims_covered, _ = llm_judge_correctness(
                row["question"], row.get("reference_answer", ""), row.get("final_answer", "")
            )
        else:
            answer_relevancy = float(row.get("decision_accuracy", 0.0) or 0.0)
            claims_covered = answer_relevancy

        hallucination = max(0.0, 1.0 - float(faithfulness_score))
        task_completion = float(row.get("decision_accuracy", 0.0) or 0.0)
        correct_answer = correct_answer_binary(
            row.get("expected_decision", "answer"),
            answer_relevancy,
            row.get("numerical_accuracy"),
            row.get("token_f1"),
            task_completion,
        )

        export_rows.append({
            "question_number":      int(row.get("question_number")),
            "retrieval_precision":  float(retrieval_precision),
            "retrieval_recall":     float(retrieval_recall),
            "retrieval_relevancy":  float(retrieval_relevancy),
            "faithfulness":         float(faithfulness_score),
            "answer_relevancy":     float(answer_relevancy),
            "hallucination":        float(hallucination),
            "token_f1":             float(row.get("token_f1") or 0.0),
            "token_precision":      float(token_precision),
            "token_recall":         float(token_recall),
            "task_completion":      float(task_completion),
            "correct_answer":       int(correct_answer),
        })

    return pd.DataFrame(export_rows)


def write_results_input_block(
    metrics_df: pd.DataFrame,
    excel_path: str,
    sheet_name: str = "Results Input",
    start_row: int = 204,
    system_name: str = "Adv RAG + LangGraph",
) -> None:
    from openpyxl import load_workbook

    wb = load_workbook(excel_path)
    ws = wb[sheet_name]
    col_map = {
        "H": "retrieval_precision",
        "I": "retrieval_recall",
        "J": "retrieval_relevancy",
        "K": "faithfulness",
        "L": "answer_relevancy",
        "M": "hallucination",
        "N": "token_f1",
        "O": "token_precision",
        "P": "token_recall",
        "Q": "task_completion",
        "R": "correct_answer",
    }

    for _, row in metrics_df.iterrows():
        target_row = start_row + int(row["question_number"]) - 1
        if ws.cell(target_row, 2).value != system_name:
            raise ValueError(
                f"Row {target_row} does not match system '{system_name}'. Found {ws.cell(target_row, 2).value!r}"
            )
        for col_letter, metric_name in col_map.items():
            ws[f"{col_letter}{target_row}"] = row[metric_name]

    wb.save(excel_path)
    print(f"Wrote {len(metrics_df)} rows to {excel_path} [{sheet_name}!H{start_row}:R{start_row + len(metrics_df) - 1}]")

LLM judge is applied during the main eval loop for every evaluated question.


In [19]:
# ---------------------------------------------------------------------------
# Evaluation loop
# ---------------------------------------------------------------------------

def run_all_pipelines(eval_df: pd.DataFrame, index: CorpusIndex) -> pd.DataFrame:
    rows = []
    sleep_sec = CONFIG["inter_question_sleep_sec"]
    prior_advanced_cache = load_prior_advanced_results(eval_df)

    for q_idx, (_, row) in enumerate(tqdm(eval_df.iterrows(), total=len(eval_df), desc="Questions")):
        question_start = time.time()
        qid               = row["id"]
        question          = row["question"]
        reference_answer  = row.get("reference_answer", "")
        company           = row.get("company", "")
        ticker            = row.get("ticker", "")
        form_type         = row.get("form_type", "")
        filing_year       = row.get("filing_year")
        evidence_chunk_id = row.get("evidence_chunk_id", "")
        context_sufficient = row.get("context_sufficient", "")
        question_number   = row.get("question_number")
        question_type     = row.get("question_type", "unknown")
        expected_decision = row.get("expected_decision", "answer")

        advanced_output = get_advanced_output(row, index, prior_advanced_cache)
        pipeline_outputs = [
            advanced_output,
            run_agentic_rag(question, index, advanced_output=advanced_output),
        ]

        for out in pipeline_outputs:
            final_answer = normalize_answer_text(out.get("final_answer", ""))
            generated_answer = normalize_answer_text(out.get("generated_answer", ""))
            predicted_decision = compute_decision_from_text(final_answer)
            decision_accuracy = 1.0 if predicted_decision == expected_decision else 0.0
            t_f1 = token_f1_score(final_answer, reference_answer) if reference_answer else None
            num_acc = numerical_accuracy_score(final_answer, reference_answer) if reference_answer else None

            retrieved_chunks = out.get("retrieved_chunks", [])
            judge_chunks, judge_chars = get_model_aware_context_limits(
                "judge", "judge", CONFIG["judge_max_context_chunks"], CONFIG["judge_max_context_chars"]
            )
            context_str = format_retrieved_context(
                retrieved_chunks,
                max_chunks=judge_chunks,
                max_chars=judge_chars,
            ) if retrieved_chunks and isinstance(retrieved_chunks, list) and len(retrieved_chunks) > 0 else final_answer

            llm_correctness_score = out.get("llm_correctness_score")
            llm_claims_covered = out.get("llm_claims_covered")
            llm_correctness_reason = out.get("llm_correctness_reason")
            judge_ran = False
            if reference_answer and (llm_correctness_score is None or pd.isna(llm_correctness_score)):
                llm_correctness_score, llm_claims_covered, llm_correctness_reason = llm_judge_correctness(
                    question, reference_answer, final_answer
                )
                judge_ran = True

            llm_faithfulness_score = out.get("llm_faithfulness_score")
            llm_faithfulness_reason = out.get("llm_faithfulness_reason")
            if llm_faithfulness_score is None or pd.isna(llm_faithfulness_score):
                llm_faithfulness_score, _, llm_faithfulness_reason = llm_judge_faithfulness(
                    question, context_str, final_answer
                )
                judge_ran = True

            judge_snapshot = get_task_model_snapshot(["judge"]) if judge_ran else {}
            judge_model_used = judge_snapshot.get("judge") if judge_snapshot else out.get("judge_llm")
            merged_llm_models = dict(out.get("llm_models_used", {}))
            if judge_ran and judge_model_used:
                merged_llm_models["judge"] = judge_model_used
                out["judge_llm"] = judge_model_used
            out["llm_models_used"] = merged_llm_models

            rows.append({
                "id":                      qid,
                "question":                question,
                "question_number":         question_number,
                "company":                 company,
                "ticker":                  ticker,
                "form_type":               form_type,
                "filing_year":             filing_year,
                "question_type":           question_type,
                "reference_answer":        reference_answer,
                "evidence_chunk_id":       evidence_chunk_id,
                "context_sufficient":      context_sufficient,
                "expected_decision":       expected_decision,
                "pipeline":                out["pipeline"],
                "output_source":           out.get("output_source", "live"),
                "rewritten_query":         out.get("rewritten_query", question),
                "retrieved_doc_names":     out.get("retrieved_doc_names", []),
                "context_retries":         out.get("context_retries", 0),
                "context_eval_relevant":   out.get("context_eval_relevant"),
                "generated_answer":        generated_answer,
                "critic_decision":         out.get("critic_decision"),
                "critic_reason":           out.get("critic_reason"),
                "repair_used":             out.get("repair_used", False),
                "repair_decision":         out.get("repair_decision"),
                "repair_reason":           out.get("repair_reason"),
                "repair_retrieval_count":  out.get("repair_retrieval_count", 0),
                "needs_decomposition":     out.get("needs_decomposition", False),
                "retrieval_sanity_failed": out.get("retrieval_sanity_failed", False),
                "drift_triggered":         out.get("drift_triggered", False),
                "drift_pending_scrapes":   out.get("drift_pending_scrapes", []),
                "ingestion_just_ran":      out.get("ingestion_just_ran", False),
                "agent_trace":             out.get("agent_trace", out.get("pipeline", "")),
                "final_answer":            final_answer,
                "latency_sec":             out.get("latency_sec"),
                "token_f1":                t_f1,
                "numerical_accuracy":      num_acc,
                "predicted_decision":      predicted_decision,
                "decision_accuracy":       decision_accuracy,
                "llm_correctness_score":   llm_correctness_score,
                "llm_correctness_reason":  llm_correctness_reason,
                "llm_claims_covered":      llm_claims_covered,
                "llm_faithfulness_score":  llm_faithfulness_score,
                "llm_faithfulness_reason": llm_faithfulness_reason,
                "retrieved_chunks":        retrieved_chunks,
                "n_chunks":                out.get("precomputed_n_chunks", len(retrieved_chunks) if isinstance(retrieved_chunks, list) else 0),
                "llm_models_used":         json.dumps(merged_llm_models, ensure_ascii=False),
                "generator_llm":           out.get("generator_llm"),
                "agent_llm":               out.get("agent_llm"),
                "judge_llm":               judge_model_used,
            })

        advanced_status = (
            out.get("output_source", "live")
            if (out := advanced_output).get("output_source") == "prior_advanced_results"
            else (out.get("generator_llm") or "-")
        )
        agentic_output = pipeline_outputs[1]
        agentic_llm = (
            agentic_output.get("generator_llm")
            or agentic_output.get("agent_llm")
            or agentic_output.get("judge_llm")
            or "-"
        )
        question_elapsed = time.time() - question_start
        if CONFIG.get("verbose_eval_logging", True):
            print(
                f"[Q {q_idx + 1}/{len(eval_df)}] {qid} | "
                f"advanced_rag={advanced_status} | "
                f"agentic_rag={agentic_llm} | "
                f"elapsed={question_elapsed:.1f}s"
            )

        if q_idx < len(eval_df) - 1:
            time.sleep(sleep_sec)

    return pd.DataFrame(rows)


results_df = run_all_pipelines(eval_df, global_index)


Loaded prior advanced results: C:\Users\jeeey\GenAI-with-LLMs\baseline_advanced_rag\results\advanced_results.csv | rows=100 | matches_by_id=80/80 | matches_by_question=80/80


Questions:   0%|          | 0/80 [00:00<?, ?it/s]

  [Planner] single | 1 sub-query
[Q 1/80] SEC_006 | advanced_rag=prior_advanced_results | agentic_rag=gemini::gemini-2.5-flash | elapsed=100.3s
  [Planner] single | 1 sub-query
[Q 2/80] SEC_007 | advanced_rag=prior_advanced_results | agentic_rag=gemini::gemini-2.5-flash | elapsed=120.0s
  [Planner] single | 1 sub-query
[Q 3/80] SEC_008 | advanced_rag=prior_advanced_results | agentic_rag=gemini::gemini-2.5-flash | elapsed=96.9s
  [RateLimit] 10 RPM cap — waiting 2.7s
  [Planner] single | 1 sub-query
  [RateLimit] 10 RPM cap — waiting 4.2s
  [RateLimit] 10 RPM cap — waiting 1.3s
  [RateLimit] 10 RPM cap — waiting 7.0s
  [RateLimit] 10 RPM cap — waiting 1.9s
  [RateLimit] 10 RPM cap — waiting 0.1s
  [RateLimit] 10 RPM cap — waiting 0.1s
[Q 4/80] SEC_009 | advanced_rag=prior_advanced_results | agentic_rag=gemini::gemini-2.5-flash | elapsed=64.9s
  [Planner] single | 1 sub-query
  [RateLimit] 10 RPM cap — waiting 1.2s
  [RateLimit] 10 RPM cap — waiting 0.6s
  [RateLimit] 10 RPM cap — waitin

In [20]:
# ---------------------------------------------------------------------------
# Metrics aggregation + display
# ---------------------------------------------------------------------------

def nanmean(series) -> float:
    vals = [v for v in series if pd.notna(v)]
    return float(np.mean(vals)) if vals else float("nan")


def rate_of(series, match_val) -> float:
    vals = [v == match_val for v in series if pd.notna(v)]
    return float(np.mean(vals)) if vals else float("nan")


def aggregate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for pipeline, grp in df.groupby("pipeline"):
        rows.append({
            "pipeline":                  pipeline,
            "n_questions":               len(grp),
            "token_f1":                  nanmean(grp["token_f1"]),
            "numerical_accuracy":        nanmean(grp["numerical_accuracy"]),
            "decision_accuracy":         nanmean(grp["decision_accuracy"]),
            "avg_latency_sec":           nanmean(grp["latency_sec"]),
            "llm_faithfulness_score":    nanmean(grp["llm_faithfulness_score"]),
            "llm_correctness_score":     nanmean(grp["llm_correctness_score"]),
            "llm_claims_covered":        nanmean(grp["llm_claims_covered"]),
            "decomposition_rate":        nanmean(grp["needs_decomposition"]) if "needs_decomposition" in grp else float("nan"),
            "critic_repair_rate":        rate_of(grp["critic_decision"], "repair"),
            "critic_insufficient_rate":  rate_of(grp["critic_decision"], "insufficient_data"),
            "repair_invocation_rate":    nanmean(grp["repair_used"]),
            "avg_context_retries":       nanmean(grp["context_retries"]),
        })
    return pd.DataFrame(rows).set_index("pipeline")


def build_aligned_results_df(results_df: pd.DataFrame) -> pd.DataFrame:
    aligned_rows = []
    for _, row in results_df.iterrows():
        final_answer = normalize_answer_text(row.get("final_answer", ""))
        context_str = format_retrieved_context(
            row.get("retrieved_chunks", []),
            max_chunks=CONFIG["judge_max_context_chunks"],
            max_chars=CONFIG["judge_max_context_chars"],
        ) if row.get("retrieved_chunks") else final_answer

        correctness = row.get("llm_correctness_score")
        claims_covered = row.get("llm_claims_covered")
        faithfulness = row.get("llm_faithfulness_score")
        corr_explanation = row.get("llm_correctness_reason", "")
        faith_explanation = row.get("llm_faithfulness_reason", "")

        if (correctness is None or pd.isna(correctness)) and row.get("reference_answer", ""):
            correctness, claims_covered, corr_explanation = llm_judge_correctness(
                row["question"],
                row.get("reference_answer", ""),
                final_answer,
            )

        if faithfulness is None or pd.isna(faithfulness):
            faithfulness, _, faith_explanation = llm_judge_faithfulness(
                row["question"],
                context_str,
                final_answer,
            )

        metadata = {
            "ticker": row.get("ticker"),
            "filing_year": None if pd.isna(row.get("filing_year")) else int(row.get("filing_year")),
            "form_type": row.get("form_type"),
        }

        refused = str(row.get("predicted_decision", "")).strip().lower() in {"warn", "refuse"}

        aligned_rows.append({
            "question_id":       row.get("id"),
            "question":          row.get("question"),
            "question_type":     row.get("question_type"),
            "expected_decision": str(row.get("expected_decision", "")).upper(),
            "expected_answer":   row.get("reference_answer", ""),
            "predicted_answer":  final_answer,
            "refused":           bool(refused),
            "correctness":       float(correctness) if correctness is not None and pd.notna(correctness) else float("nan"),
            "faithfulness":      float(faithfulness) if faithfulness is not None and pd.notna(faithfulness) else float("nan"),
            "claims_covered":    float(claims_covered) if claims_covered is not None and pd.notna(claims_covered) else float("nan"),
            "n_chunks":          int(row.get("n_chunks", len(row.get("retrieved_chunks", []) or [])) or 0),
            "rewritten_query":   row.get("rewritten_query", row.get("question")),
            "metadata":          str(metadata),
            "corr_explanation":  corr_explanation,
            "faith_explanation": faith_explanation,
            "pipeline":          row.get("pipeline"),
        })
    return pd.DataFrame(aligned_rows)


def print_advanced_style_summary(results_subset: pd.DataFrame, title: str) -> None:
    print("=" * 60)
    print(f" {title}")
    print("=" * 60)

    scored_c = results_subset[results_subset["llm_correctness_score"].notna()]
    scored_f = results_subset[results_subset["llm_faithfulness_score"].notna()]
    print(f"\nTotal questions    : {len(results_subset)}")
    print(f"Correctness judged : {len(scored_c)}")
    print(f"Faithfulness judged: {len(scored_f)}")

    print("\nOverall metrics:")
    for col, label in [
        ("llm_correctness_score",  "LLM Correctness  "),
        ("llm_claims_covered",     "Claims Covered   "),
        ("llm_faithfulness_score", "LLM Faithfulness "),
        ("token_f1",               "Token F1         "),
        ("numerical_accuracy",     "Numerical Accuracy"),
        ("decision_accuracy",      "Decision Accuracy"),
    ]:
        if col not in results_subset.columns:
            continue
        vals = results_subset[col].dropna()
        if len(vals):
            print(f"  {label}: {vals.mean():.4f}  (n={len(vals)})")

    print("\nBy question_type:")
    type_cols = ["llm_correctness_score", "llm_faithfulness_score", "token_f1", "decision_accuracy"]
    avail = [c for c in type_cols if c in results_subset.columns]
    if avail and "question_type" in results_subset.columns and len(results_subset):
        print(results_subset.groupby("question_type")[avail].mean().round(3).to_string())
    else:
        print("(No question_type breakdown available)")

    if "latency_sec" in results_subset.columns:
        lat = results_subset["latency_sec"].dropna()
        if len(lat):
            print(f"\nLatency: mean={lat.mean():.1f}s  median={lat.median():.1f}s  max={lat.max():.1f}s")

    token_cols = [
        "tokens_generate_input", "tokens_generate_output",
        "tokens_judge_corr_input", "tokens_judge_corr_output",
        "tokens_judge_faith_input", "tokens_judge_faith_output",
        "tokens_total_input", "tokens_total_output",
    ]
    present_token_cols = [c for c in token_cols if c in results_subset.columns]
    if present_token_cols or "est_cost_usd" in results_subset.columns:
        print("\nToken & cost summary:")
        for col in present_token_cols:
            print(f"  {col:35s}: {int(results_subset[col].fillna(0).sum()):,}")
        if "est_cost_usd" in results_subset.columns:
            print(f"  {'Total estimated cost (USD)':35s}: ${results_subset['est_cost_usd'].fillna(0).sum():.4f}")


def print_pipeline_comparison(results_df: pd.DataFrame, left_pipeline: str = "advanced_rag", right_pipeline: str = "agentic_rag") -> None:
    left_df = results_df[results_df["pipeline"] == left_pipeline].copy()
    right_df = results_df[results_df["pipeline"] == right_pipeline].copy()
    if left_df.empty or right_df.empty:
        print("\n(Pipeline comparison unavailable: one or both pipeline result sets are empty)")
        return

    print("\n" + "=" * 60)
    print(f" COMPARISON: {left_pipeline.upper()} vs {right_pipeline.upper()}")
    print("=" * 60)
    cols = ["llm_correctness_score", "llm_faithfulness_score", "token_f1", "decision_accuracy"]
    avail = [c for c in cols if c in left_df.columns and c in right_df.columns]
    comparison = pd.DataFrame({
        left_pipeline: left_df[avail].mean(),
        right_pipeline: right_df[avail].mean(),
    }).round(3)
    comparison["Delta"] = (comparison[right_pipeline] - comparison[left_pipeline]).round(3)
    print(comparison.to_string())


def build_advanced_style_export(results_subset: pd.DataFrame) -> pd.DataFrame:
    export_rows = []
    for _, row in results_subset.iterrows():
        export_rows.append({
            "question_id": row.get("id"),
            "question": row.get("question"),
            "question_type": row.get("question_type"),
            "company": row.get("company", ""),
            "ticker": row.get("ticker", ""),
            "form_type": row.get("form_type", ""),
            "filing_year": row.get("filing_year"),
            "expected_decision": row.get("expected_decision"),
            "expected_answer": row.get("reference_answer", ""),
            "predicted_answer": row.get("final_answer", ""),
            "refused": str(row.get("predicted_decision", "")).strip().lower() in {"warn", "refuse"},
            "correctness": row.get("llm_correctness_score"),
            "faithfulness": row.get("llm_faithfulness_score"),
            "claims_covered": row.get("llm_claims_covered"),
            "n_chunks": row.get("n_chunks", 0),
            "rewritten_query": row.get("rewritten_query", row.get("question", "")),
            "metadata": str({
                "ticker": row.get("ticker"),
                "filing_year": None if pd.isna(row.get("filing_year")) else int(row.get("filing_year")),
                "form_type": row.get("form_type"),
            }),
            "corr_explanation": row.get("llm_correctness_reason", ""),
            "faith_explanation": row.get("llm_faithfulness_reason", ""),
            "latency_sec": row.get("latency_sec"),
            "token_f1": row.get("token_f1"),
            "numerical_accuracy": row.get("numerical_accuracy"),
            "decision_accuracy": row.get("decision_accuracy"),
            "output_source": row.get("output_source", "live"),
        })
    return pd.DataFrame(export_rows)


summary_df = aggregate_metrics(results_df)
aligned_results_df = build_aligned_results_df(results_df)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

for pipeline_name, title in [
    ("advanced_rag", "ADVANCED RAG — EVALUATION RESULTS"),
    ("agentic_rag", "AGENTIC RAG — EVALUATION RESULTS"),
]:
    pipeline_df = results_df[results_df["pipeline"] == pipeline_name].copy()
    if not pipeline_df.empty:
        print_advanced_style_summary(pipeline_df, title)

print_pipeline_comparison(results_df, "advanced_rag", "agentic_rag")

print("\n=== Summary metrics by pipeline ===")
display(summary_df.T)

print("\n=== Aligned final output metrics (advanced-RAG-style export schema) ===")
display(aligned_results_df.head())

comparison_cols = ["correctness", "faithfulness", "claims_covered"]
aligned_comparison_df = (
    aligned_results_df
    .groupby("pipeline")[comparison_cols]
    .mean()
    .round(3)
)
print("\n=== Pipeline comparison on aligned metrics ===")
display(aligned_comparison_df)

results_dir = Path(CONFIG["results_dir"])
results_dir.mkdir(parents=True, exist_ok=True)
for pipeline_name in results_df["pipeline"].unique():
    pipeline_eval_df = results_df[results_df["pipeline"] == pipeline_name].reset_index(drop=True)
    advanced_style_df = build_advanced_style_export(pipeline_eval_df)
    eval_out_path = results_dir / f"{pipeline_name}_eval_results.csv"
    advanced_style_out_path = results_dir / f"{pipeline_name}_advanced_style_results.csv"
    aligned_out_path = results_dir / f"{pipeline_name}_aligned_results.csv"

    pipeline_eval_df.to_csv(eval_out_path, index=False)
    advanced_style_df.to_csv(advanced_style_out_path, index=False)
    aligned_results_df[aligned_results_df["pipeline"] == pipeline_name].drop(columns=["pipeline"]).reset_index(drop=True).to_csv(aligned_out_path, index=False)

    print(f"Saved {len(pipeline_eval_df)} rows -> {eval_out_path}")
    print(f"Saved {len(advanced_style_df)} rows -> {advanced_style_out_path}")
    print(f"Saved {len(advanced_style_df)} rows -> {aligned_out_path}")

# Show agentic-only per-question detail
agentic_rows = results_df[results_df["pipeline"] == "agentic_rag"].copy()
display_cols = [
    "id", "question_type", "question", "reference_answer",
    "generated_answer", "final_answer", "rewritten_query",
    "needs_decomposition", "context_retries", "critic_decision", "critic_reason",
    "repair_used", "repair_decision", "repair_reason", "repair_retrieval_count",
    "retrieval_sanity_failed", "drift_triggered", "ingestion_just_ran",
    "drift_pending_scrapes", "agent_trace", "output_source",
]
print("\n=== Agentic RAG -- per-question detail (LLM answer vs gold answer + route trace) ===")
display(agentic_rows[display_cols].reset_index(drop=True))

if CONFIG.get("auto_export_results_input", False):
    metrics_export_df = build_results_input_block(results_df, pipeline_name="agentic_rag")
    write_results_input_block(
        metrics_export_df,
        CONFIG["results_input_excel_path"],
        sheet_name="Results Input",
        start_row=204,
        system_name="Adv RAG + LangGraph",
    )


 ADVANCED RAG — EVALUATION RESULTS

Total questions    : 80
Correctness judged : 60
Faithfulness judged: 80

Overall metrics:
  LLM Correctness  : 0.0958  (n=60)
  Claims Covered   : 0.1045  (n=60)
  LLM Faithfulness : 1.0000  (n=80)
  Token F1         : 0.1794  (n=60)
  Numerical Accuracy: 0.3833  (n=60)
  Decision Accuracy: 0.7500  (n=80)

By question_type:
               llm_correctness_score  llm_faithfulness_score  token_f1  decision_accuracy
question_type                                                                            
adversarial                      NaN                   1.000       NaN              0.000
extractive                     0.050                   1.000     0.199              1.000
paraphrased                    0.238                   1.000     0.202              1.000
reasoning                      0.000                   1.000     0.138              1.000

Latency: mean=4.9s  median=5.2s  max=6.1s
 AGENTIC RAG — EVALUATION RESULTS

Total questions    :

pipeline,advanced_rag,agentic_rag
n_questions,80.000,80.000
token_f1,0.179,0.433
numerical_accuracy,0.383,0.883
decision_accuracy,0.750,0.512
avg_latency_sec,4.918,67.564
llm_faithfulness_score,1.000,0.849
llm_correctness_score,0.096,0.486
llm_claims_covered,0.104,0.496
decomposition_rate,0.000,0.037
critic_repair_rate,NaN,0.039



=== Aligned final output metrics (advanced-RAG-style export schema) ===


,question_id,question,question_type,expected_decision,expected_answer,predicted_answer,refused,correctness,faithfulness,claims_covered,n_chunks,rewritten_query,metadata,corr_explanation,faith_explanation,pipeline
0,SEC_006,What was Nvidia's gross margin percentage in f...,extractive,ANSWER,"Nvidia's gross margin was 72.7% in FY2024, up ...",Insufficient information in the provided filings.,False,0.000,1.000,0.000,0,N,"{'ticker': 'NVDA', 'filing_year': 2024, 'form_...",The candidate answer incorrectly states that t...,The candidate answer accurately reflects the p...,advanced_rag
1,SEC_006,What was Nvidia's gross margin percentage in f...,extractive,ANSWER,"Nvidia's gross margin was 72.7% in FY2024, up ...",Insufficient data: The required information co...,True,0.000,0.000,0.000,7,Nvidia gross margin percentage for fiscal year...,"{'ticker': 'NVDA', 'filing_year': 2024, 'form_...",The candidate incorrectly states that the data...,The candidate answer correctly states that the...,agentic_rag
2,SEC_007,How many employees did Nvidia report at the en...,extractive,ANSWER,"As of the end of fiscal year 2025, Nvidia had ...",Insufficient information in the provided filin...,False,0.000,1.000,0.000,7,Nvidia total employees,"{'ticker': 'NVDA', 'filing_year': 2025, 'form_...",The candidate answer states 'Insufficient info...,The candidate answer directly states that ther...,advanced_rag
3,SEC_007,How many employees did Nvidia report at the en...,extractive,ANSWER,"As of the end of fiscal year 2025, Nvidia had ...",The retrieved context does not contain informa...,False,0.000,1.000,0.000,7,NVIDIA (NVDA) total number of employees for fi...,"{'ticker': 'NVDA', 'filing_year': 2025, 'form_...",The candidate answer provides information for ...,The candidate correctly states that the inform...,agentic_rag
4,SEC_008,What was JPMorgan's CET1 capital ratio at the ...,extractive,ANSWER,JPMorgan's Standardized and Advanced CET1 capi...,Insufficient information in the provided filings.,False,0.000,1.000,0.000,7,JPMorgan Chase &,"{'ticker': 'JPM', 'filing_year': 2023, 'form_t...",The candidate incorrectly states that informat...,The candidate answer accurately reflects the p...,advanced_rag



=== Pipeline comparison on aligned metrics ===


,correctness,faithfulness,claims_covered
pipeline,,,
advanced_rag,0.096,1.000,0.104
agentic_rag,0.486,0.849,0.496


Saved 80 rows -> results\advanced_rag_eval_results.csv
Saved 80 rows -> results\advanced_rag_advanced_style_results.csv
Saved 80 rows -> results\advanced_rag_aligned_results.csv
Saved 80 rows -> results\agentic_rag_eval_results.csv
Saved 80 rows -> results\agentic_rag_advanced_style_results.csv
Saved 80 rows -> results\agentic_rag_aligned_results.csv

=== Agentic RAG -- per-question detail (LLM answer vs gold answer + route trace) ===


,id,question_type,question,reference_answer,generated_answer,final_answer,rewritten_query,needs_decomposition,context_retries,critic_decision,critic_reason,repair_used,repair_decision,repair_reason,repair_retrieval_count,retrieval_sanity_failed,drift_triggered,ingestion_just_ran,drift_pending_scrapes,agent_trace,output_source
0,SEC_006,extractive,What was Nvidia's gross margin percentage in f...,"Nvidia's gross margin was 72.7% in FY2024, up ...",Nvidia's gross margin percentage in fiscal yea...,Insufficient data: The required information co...,Nvidia gross margin percentage for fiscal year...,False,0,insufficient_data,The context provides gross margin percentages ...,False,NaN,NaN,0,False,False,False,[],planner -> retrieve -> generate -> critic=insu...,seeded_prior_advanced
1,SEC_007,extractive,How many employees did Nvidia report at the en...,"As of the end of fiscal year 2025, Nvidia had ...",The retrieved context does not contain informa...,The retrieved context does not contain informa...,NVIDIA (NVDA) total number of employees for fi...,False,1,accept,The answer correctly states that the informati...,False,NaN,NaN,0,False,False,False,[],planner -> retrieve -> context_retry=1 -> gene...,seeded_prior_advanced
2,SEC_008,extractive,What was JPMorgan's CET1 capital ratio at the ...,JPMorgan's Standardized and Advanced CET1 capi...,The retrieved context does not contain JPMorga...,Insufficient data: The required information co...,JPMorgan Chase & Co. Common Equity Tier 1 (CET...,False,1,insufficient_data,The retrieved context provides CET1 capital ra...,False,NaN,NaN,0,False,False,False,[],planner -> retrieve -> context_retry=1 -> gene...,seeded_prior_advanced
3,SEC_009,extractive,Which federal regulator supervises JPMorgan Ch...,JPMorgan Chase is supervised and regulated by ...,The Federal Reserve supervises JPMorgan Chase ...,The Federal Reserve supervises JPMorgan Chase ...,JPMorgan Chase bank holding company federal re...,False,0,accept,The answer correctly identifies the Federal Re...,False,NaN,NaN,0,False,False,False,[],planner -> retrieve -> generate -> critic=accept,seeded_prior_advanced
4,SEC_010,extractive,What were Bank of America's Total liabilities ...,Bank of America's total liabilities as of Sept...,Bank of America's Total liabilities as of Sept...,Bank of America's Total liabilities as of Sept...,Bank of America (BAC) total liabilities as of ...,False,0,accept,The answer accurately states Bank of America's...,False,NaN,NaN,0,False,False,False,[],planner -> retrieve -> generate -> critic=accept,seeded_prior_advanced
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,SEC_096,adversarial,What internal AI system does Chevron use for d...,,The provided context does not contain informat...,The provided context does not contain informat...,Chevron artificial intelligence systems for dr...,False,1,accept,The answer correctly states that the context d...,False,NaN,NaN,0,False,False,False,[],planner -> retrieve -> context_retry=1 -> gene...,seeded_prior_advanced
76,SEC_097,adversarial,What percentage of Walmart employees work remo...,,The provided context does not contain informat...,The provided context does not contain informat...,Walmart global remote employee percentage,False,1,accept,The provided context does not contain any info...,False,NaN,NaN,0,False,False,False,[],planner -> retrieve -> context_retry=1 -> gene...,seeded_prior_advanced
77,SEC_098,adversarial,What internal robot efficiency metrics does Wa...,,The provided context does not contain informat...,The provided context does not contain informat...,Walmart WMT warehouse automation robot efficie...,False,1,accept,The answer correctly states that the context d...,False,NaN,NaN,0,False,False,False,[],planner -> retrieve -> context_retry=1 -> gene...,seeded_prior_advanced
78,SEC_099,adversarial,What percentage of Costco warehouses are fully...,,The provided context does not contain informat...,The provided context does